# Hybrid quantum–classical OCT classification — notebook 02: circuit & kernel benchmark, IBM hardware, external validation

Companion code for the paper. This notebook reproduces **Sections B–F** of the results (see `README.md` for the full map).
It starts from the frozen ResNet-18d features written by **notebook 01** (`features.npz` and the backbone checkpoint in the
results folder); if they are not found, it fine-tunes the backbone and extracts them itself, with the identical split and settings.

| Experiment | Quantum model | Matched classical control |
|---|---|---|
| A. Data re-uploading VQC | 4 × 5 qubits, inputs re-encoded every layer with trainable scales, L = 3 and L = 6 | Per-group tanh MLP with **exactly** the same parameter count (240 / 480) |
| B. Quantum-kernel SVM | IQP (ZZ) fidelity kernel and **projected** quantum kernel on n-qubit PCA features, bandwidth tuned | RBF-SVM on the **same** PCA features; RBF-SVM / logistic regression on full features as reference |
| C. Geometric difference (Huang et al., *Nat. Commun.* 12, 2631, 2021) | g(K_C‖K_Q) and model complexities s_C, s_Q on the real labels | — |
| IBM hardware | the trained 4 × 5-qubit VQC head, 300 locked-test images, unmitigated (Sampler + DD) and mitigated (Estimator: TREX + gate twirling + DD) | exact simulation and the parameter-matched classical head on the same images |
| External validation | all heads applied unchanged to Duke/Srinivasan 2014 and OCTDL | same |

**Protocol.** The variants and comparisons are fixed before the locked test is opened; all locked-test comparisons are
Holm-corrected as one family. Paired tests on images that share a patient use the patient-clustered McNemar test and a
patient-cluster bootstrap.

**Run switches** (Cell 4):

* `RUN_BENCHMARK = True` runs experiments A–C and the locked/official test (≈ 1–2 h on a T4). `False` skips them (the hardware
  and external sections only need the backbone features).
* `IBM_STEP = 'none'` (default) never contacts IBM: the hardware analysis cells re-analyse the saved hardware results in the
  results folder. To repeat the hardware experiment, go `'check'` (no QPU time) → `'test'` → `'main'` → `'mit_test'` → `'mitigated'`.
  `IBM_FAKE = True` runs the same pipeline on a local fake backend without an IBM account.
* `RUN_EXTERNAL = True` runs the external validation (datasets: see the markdown above Cell 27).
* Environment variable `FAST_DEV_RUN=1` runs the whole notebook on a tiny subset in minutes (pipeline check only; the numbers are meaningless).

Qiskit runs in a **separate process** (`ibm_run.py`, written by Cell 23), because importing it next to PyTorch / scikit-learn on
Kaggle fails with *"cannot allocate memory in static TLS block"*.

In [ ]:
# Cell 0: Google Colab setup (does nothing on Kaggle or a local machine)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/oct_qml/results'    # persists across disconnects
    os.environ['EXT_DIR'] = '/content/drive/MyDrive/oct_qml/external'      # put external datasets (zips) here
    for d in [os.environ['RESULTS_DIR'], os.environ['EXT_DIR']]:
        os.makedirs(d, exist_ok=True)
    for k in ['KAGGLE_API_TOKEN', 'IBM_QUANTUM_TOKEN', 'IBM_QUANTUM_INSTANCE']:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            print(f'Colab secret {k} not set' + (' (optional)' if k == 'IBM_QUANTUM_INSTANCE' else ''))
    if os.environ.get('KAGGLE_API_TOKEN'):
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/access_token', 'w') as f:
            f.write(os.environ['KAGGLE_API_TOKEN'])
        os.chmod('/root/.kaggle/access_token', 0o600)
    get_ipython().system('pip install -q -U kaggle kagglehub')
    try:
        import kagglehub
        path = kagglehub.dataset_download('paultimothymooney/kermany2018')
    except Exception as e:
        print('kagglehub failed (', e, ') -> using the kaggle CLI')
        get_ipython().system('kaggle datasets download -d paultimothymooney/kermany2018 -p /content/data')
        get_ipython().system('unzip -q -o /content/data/kermany2018.zip -d /content/data')
        path = '/content/data'
    roots = [d for d, sub, _ in os.walk(path) if {'train', 'test'} <= set(sub) and '__MACOSX' not in d
             and all(os.path.isdir(os.path.join(d, 'train', c)) for c in ['NORMAL', 'CNV', 'DME', 'DRUSEN'])]
    assert roots, f'Kermany train/test folders not found under {path}'
    os.environ['OCT_ROOT'] = sorted(roots, key=len)[0]
    print('dataset:', repr(os.environ['OCT_ROOT']), '| results ->', os.environ['RESULTS_DIR'])
else:
    print('not in Colab -> Kaggle input, or OCT_ROOT / RESULTS_DIR / EXT_DIR environment variables, are used')

In [ ]:
# Cell 1: dependencies (PennyLane is used only to verify the simulator and draw the circuit)
!pip install -q pennylane timm

In [ ]:
# Cell 2: imports, configuration, reproducibility
import os, glob, json, math, time, copy, random, hashlib, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
FAST_DEV_RUN = bool(int(os.environ.get('FAST_DEV_RUN', '0')))   # tiny pipeline smoke test only
UNLOCK_TEST = True   # the reported run: the test sets are opened once, by the final evaluation cells

GLOBAL_SEED = 42
CLASSES = ['NORMAL', 'CNV', 'DME', 'DRUSEN']
K = len(CLASSES)

CFG = dict(
    backbone='resnet18d.ra2_in1k', pretrained=True, img_size=224,
    # patient-level partition of the official *train* folder
    locked_test_frac=0.15, backbone_frac=0.50,          # remainder -> head pool (CV)
    bb_val_frac=0.10,                                    # of backbone pool, for early stopping
    # backbone fine-tuning
    bb_epochs=12, bb_patience=3, bb_bs=64, bb_lr_head=1e-3, bb_lr_body=2e-4, bb_wd=0.05,
    bb_samples_per_epoch=40000, label_smoothing=0.1, reuse_backbone=True,
    # quantum layer (HQNN-Parallel)
    q_circuits=4, q_qubits=5, q_layers=2, q_dropout=0.1,
    # heads
    n_folds=5, head_seeds=[0, 1, 2], head_epochs=40, head_bs=256, head_lr=3e-3,
    head_wd=1e-4, head_patience=6, head_in_dropout=0.2, head_min_steps=0,
    # experiments
    data_eff_per_class=[25, 100, 400, 1600], data_eff_min_steps=800,
    img_noise_levels=[0.0, 0.05, 0.10, 0.20, 0.30], noise_eval_per_class=500,
    shot_levels=[4096, 1024, 256, 64, 16], shot_repeats=5,
    depol_levels=[0.0, 1e-3, 5e-3, 1e-2, 2e-2, 5e-2],
    n_boot=2000, num_workers=4,
)
if FAST_DEV_RUN:
    CFG.update(bb_epochs=1, bb_samples_per_epoch=256, bb_bs=16, head_epochs=3, head_min_steps=300, head_seeds=[0],
               n_folds=3, data_eff_per_class=[5, 10], data_eff_min_steps=20,
               noise_eval_per_class=8, img_noise_levels=[0.0, 0.2], shot_levels=[64],
               shot_repeats=1, depol_levels=[0.0, 1e-2], n_boot=50, num_workers=0,
               reuse_backbone=False)

OUT = os.environ.get('RESULTS_DIR') or ('/kaggle/working/results' if os.path.isdir('/kaggle/working') else os.path.abspath('results'))
os.makedirs(OUT, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # speed; head experiments are seeded separately
    torch.backends.cudnn.benchmark = True
seed_everything(GLOBAL_SEED)

print('device:', DEVICE, '| AMP:', AMP, '| FAST_DEV_RUN:', FAST_DEV_RUN, '| UNLOCK_TEST:', UNLOCK_TEST)
if DEVICE.type != 'cuda' and not FAST_DEV_RUN:
    print('WARNING: no GPU. Enable a GPU accelerator (T4/P100) -- backbone fine-tuning on CPU is not viable.')
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)

In [ ]:
# Cell 3: settings for the experiments
CFG.update(
    reupload_layers=[3, 6],                         # classical control depth = 2L/3 -> exact parameter match
    kern_qubits=[8, 12], kern_train_n=4000, kern_inner_val=0.2,
    kern_bandwidths=[0.05, 0.1, 0.2, 0.4, 0.8], kern_C=[0.1, 1.0, 10.0],
    rbf_gamma_scale=[0.1, 0.3, 1.0, 3.0, 10.0],     # gamma = scale / n_features (inputs standardised)
    proj_gamma_scale=[0.3, 1.0, 3.0],
    geo_n=1500, geo_lambdas=[1e-3, 1e-2, 1e-1, 1.0],
)
if FAST_DEV_RUN:
    CFG.update(reupload_layers=[3], kern_qubits=[4], kern_train_n=80, kern_bandwidths=[0.1, 0.4],
               kern_C=[1.0], rbf_gamma_scale=[1.0], proj_gamma_scale=[1.0], geo_n=60)
print({k: CFG[k] for k in ['reupload_layers', 'kern_qubits', 'kern_train_n', 'geo_n']})

In [ ]:
# Cell 4: run switches
RUN_BENCHMARK = True     # experiments A-C: CV, kernel SVMs, geometric difference, locked + official test (~1-2 h on a T4)
IBM_STEP = 'none'        # 'none' = analysis only (nothing sent to IBM) | 'check' (no QPU time) -> 'test' -> 'main' (unmitigated) -> 'mit_test' -> 'mitigated' | 'resume'
IBM_JOB_ID = ''          # only for IBM_STEP = 'resume' (works for any earlier job, Sampler or Estimator)
IBM_MIT_N = 0            # images in the mitigated run (0 = all 300; e.g. 150 to halve QPU time)
IBM_ZNE = False          # True -> also zero-noise extrapolation (resilience level 2), roughly 3x the QPU time
RUN_EXTERNAL = True      # external validation on OCTDL / Duke-Srinivasan (no QPU time; needs the backbone checkpoint)
EXT_SEEDS = [0, 1, 2]    # head ensembles for external validation: 5 folds x these seeds
IBM_BACKEND = 'ibm_kingston'
IBM_N_IMAGES = 300       # class-balanced locked-test subset (IBM_N_IMAGES / 4 per class)
IBM_FAKE = False         # True -> local FakeTorino noisy simulator instead of IBM (no account needed; slow on CPU)
print(dict(RUN_BENCHMARK=RUN_BENCHMARK, IBM_STEP=IBM_STEP, IBM_BACKEND=IBM_BACKEND, IBM_N_IMAGES=IBM_N_IMAGES, IBM_MIT_N=IBM_MIT_N, IBM_ZNE=IBM_ZNE, IBM_FAKE=IBM_FAKE, RUN_EXTERNAL=RUN_EXTERNAL))

In [ ]:
# Cell 5: paired tests that respect clustering (several images per patient / volume)
def clustered_mcnemar(correct_a, correct_b, groups):
    # Durkalski et al. (Stat Med 2003): McNemar test for clustered matched-pair data
    a_only = (correct_a & ~correct_b).astype(int); b_only = (~correct_a & correct_b).astype(int)
    d = pd.DataFrame({'g': groups, 'a': a_only, 'b': b_only}).groupby('g').sum()
    diff = (d.a - d.b).values.astype(float)
    if (diff ** 2).sum() == 0:
        return 1.0
    return float(stats.chi2.sf(diff.sum() ** 2 / (diff ** 2).sum(), 1))

def boot_p(deltas):
    # two-sided percentile-bootstrap p value for H0: delta = 0
    deltas = np.asarray(deltas)
    return float(min(1.0, 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())))
print('clustered McNemar / bootstrap helpers ready')

In [ ]:
# Cell 6: index images, remove patients shared with the official test set, patient-level partition
def find_root():
    # Search /kaggle/input (any depth) for a folder holding train/ and test/ with the class subfolders.
    env = os.environ.get('OCT_ROOT')
    if env and os.path.isdir(os.path.join(env, 'train')):
        return env
    base = '/kaggle/input'
    hits = []
    for dirpath, dirnames, _ in os.walk(base):
        if dirpath[len(base):].count(os.sep) > 6:
            dirnames[:] = []; continue
        if 'train' in dirnames and 'test' in dirnames and \
                all(os.path.isdir(os.path.join(dirpath, 'train', c)) for c in CLASSES):
            hits.append(dirpath); dirnames[:] = []
    if hits:
        hits.sort(key=lambda p: ('OCT' not in p, len(p)))   # prefer the OCT2017 folder over duplicates
        if len(hits) > 1:
            print('several candidate roots found, using the first:', hits)
        return hits[0]
    print('Nothing usable under /kaggle/input. What is attached:')
    if os.path.isdir(base):
        for dirpath, dirnames, files in os.walk(base):
            depth = dirpath[len(base):].count(os.sep)
            if depth > 3:
                dirnames[:] = []; continue
            print('  ' * depth + os.path.basename(dirpath) + '/', f'({len(files)} files)')
    else:
        print('  /kaggle/input does not exist')
    raise FileNotFoundError(
        'Kermany OCT2017 not found. In the Kaggle editor: Add Input -> search "kermany2018" '
        '(paultimothymooney) -> Add, then re-run this cell. Or set OCT_ROOT to the folder containing train/ and test/.')
ROOT = find_root(); print('data root:', repr(ROOT))

def patient_id(path):            # Kermany filenames: CLASS-PATIENTID-N.jpeg
    return os.path.basename(path).split('-')[1]

def strat_sample(df, n_per_class, seed):
    parts = [d.sample(min(len(d), n_per_class), random_state=seed) for _, d in df.groupby('label')]
    return pd.concat(parts).sort_index().reset_index(drop=True)

def index_split(split):
    rows = []
    for lab, c in enumerate(CLASSES):
        for f in sorted(glob.glob(os.path.join(ROOT, split, c, '*.jpeg'))):
            rows.append((f, lab, patient_id(f)))
    return pd.DataFrame(rows, columns=['path', 'label', 'patient'])

train_all = index_split('train')
official = index_split('test')

if FAST_DEV_RUN:
    train_all = strat_sample(train_all, 120, GLOBAL_SEED)
    official = strat_sample(official, 20, GLOBAL_SEED)

shared = set(train_all.patient) & set(official.patient)
print(f'train folder: {len(train_all)} imgs / {train_all.patient.nunique()} patients; '
      f'official test: {len(official)} imgs / {official.patient.nunique()} patients')
print(f'patients present in BOTH train folder and official test: {len(shared)} '
      f'-> {train_all.patient.isin(shared).sum()} train-folder images removed')
train_all = train_all[~train_all.patient.isin(shared)].reset_index(drop=True)

# Patient-level partition, stratified by each patient's majority label.
# A patient can have scans in several classes; the patient (not the image) is the unit.
pat_label = train_all.groupby('patient')['label'].agg(lambda s: s.value_counts().idxmax())
rng = np.random.RandomState(GLOBAL_SEED)
assign = {}
for lab, grp in pat_label.groupby(pat_label):
    pats = np.array(sorted(grp.index)); rng.shuffle(pats)
    n = len(pats)
    a = int(round(n * CFG['locked_test_frac'])); b = a + int(round(n * CFG['backbone_frac']))
    for p in pats[:a]:  assign[p] = 'locked_test'
    for p in pats[a:b]: assign[p] = 'backbone'
    for p in pats[b:]:  assign[p] = 'head'
train_all['pool'] = train_all.patient.map(assign)

# backbone-val (early stopping only) = patient-disjoint slice of the backbone pool
bb_pats = pat_label[pat_label.index.isin(train_all.loc[train_all.pool == 'backbone', 'patient'])]
for lab, grp in bb_pats.groupby(bb_pats):
    pats = np.array(sorted(grp.index)); rng.shuffle(pats)
    for p in pats[:max(1, int(round(len(pats) * CFG['bb_val_frac'])))]:
        assign[p] = 'backbone_val'
train_all['pool'] = train_all.patient.map(assign)

POOLS = {k: train_all[train_all.pool == k].reset_index(drop=True)
         for k in ['backbone', 'backbone_val', 'head', 'locked_test']}
POOLS['official_test'] = official.reset_index(drop=True)

names = list(POOLS)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        assert not set(POOLS[a].patient) & set(POOLS[b].patient), f'patient overlap {a}/{b}'
        assert not set(POOLS[a].path) & set(POOLS[b].path), f'image overlap {a}/{b}'

summary = pd.DataFrame({k: v.label.value_counts().reindex(range(K), fill_value=0).values
                        for k, v in POOLS.items()}, index=CLASSES).T
summary['images'] = summary.sum(axis=1); summary['patients'] = [v.patient.nunique() for v in POOLS.values()]
print(summary.to_string()); print('All pools are image- and patient-disjoint.')

locked_hash_full = hashlib.sha256('\n'.join(sorted(POOLS['locked_test'].path.map(os.path.basename))).encode()).hexdigest()   # path-independent
locked_hash = locked_hash_full[:16]
print('locked-test SHA-256 (filenames only):', locked_hash_full)
print('  -> paper uses the first 16 characters:', locked_hash)
json.dump({k: sorted(v.path.map(os.path.basename)) for k, v in POOLS.items()},
          open(f'{OUT}/split_manifest.json', 'w'))
summary.to_csv(f'{OUT}/split_summary.csv')

In [ ]:
# Cell 7: transforms, pixel-space noise, dataset
_cfg = timm.data.resolve_model_data_config(timm.create_model(CFG['backbone'], pretrained=False))
MEAN, STD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))  # grayscale -> one mean/std
S = CFG['img_size']

train_tf = T.Compose([
    T.RandomResizedCrop(S, scale=(0.80, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2), T.ToTensor()])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor()])

def add_pixel_noise(x, kind, level, gen):
    # x: (1,H,W) in [0,1] BEFORE normalisation. gaussian: x + s*n ; speckle (multiplicative): x*(1 + s*n)
    if level <= 0:
        return x
    n = torch.randn(x.shape, generator=gen)
    x = x + level * n if kind == 'gaussian' else x * (1.0 + level * n)
    return x.clamp(0.0, 1.0)

class OCTDataset(Dataset):
    def __init__(self, df, train=False, noise=None):
        self.paths, self.labels = df.path.tolist(), df.label.tolist()
        self.tf = train_tf if train else eval_tf
        self.noise = noise                       # None or (kind, level)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        x = self.tf(Image.open(self.paths[i]).convert('L'))
        if self.noise is not None:
            # same noise field per image across levels -> only the amplitude changes
            g = torch.Generator().manual_seed(GLOBAL_SEED * 1_000_003 + i)
            x = add_pixel_noise(x, *self.noise, g)
        x = (x - MEAN) / STD
        return x.expand(3, -1, -1), self.labels[i]

def make_loader(df, train=False, bs=128, sampler=None, noise=None):
    return DataLoader(OCTDataset(df, train, noise), batch_size=bs, sampler=sampler,
                      shuffle=False, num_workers=CFG['num_workers'], pin_memory=DEVICE.type == 'cuda',
                      drop_last=train, persistent_workers=CFG['num_workers'] > 0)

def metrics(y, prob):
    pred = np.asarray(prob).argmax(1)
    return dict(acc=accuracy_score(y, pred), bacc=balanced_accuracy_score(y, pred),
                f1=f1_score(y, pred, average='macro'))
print('ok')

In [ ]:
# Cell 8: reuse the frozen-backbone features from notebook 01 (features.npz) if they match this split
def _find(pattern):
    hits = []
    for base in [OUT, '/kaggle/input']:
        if os.path.isdir(base):
            hits += glob.glob(os.path.join(base, '**', pattern), recursive=True)
    if os.environ.get('FEATURES_NPZ') and pattern == 'features.npz':
        hits.insert(0, os.environ['FEATURES_NPZ'])
    return hits

FEAT, HAVE_FEATS = {}, False
for f in _find('features.npz'):
    z = np.load(f)
    ok = all(f'{k}_X' in z and len(z[f'{k}_y']) == len(POOLS[k]) and (z[f'{k}_y'] == POOLS[k].label.values).all()
             for k in ['head', 'locked_test', 'official_test'])
    if not ok:
        print('skipping', f, '(does not match this split -- different dataset root or settings)'); continue
    for k in ['head', 'locked_test', 'official_test']:
        FEAT[k] = dict(X=z[f'{k}_X'].astype(np.float32), bb_prob=z[f'{k}_bb_prob'],
                       y=POOLS[k].label.values, g=POOLS[k].patient.values)
    HAVE_FEATS = True; print('loaded backbone features from', f); break

if not HAVE_FEATS:
    for c in _find('backbone_*.pt'):
        if not c.startswith(OUT):
            import shutil; shutil.copy(c, OUT); print('found backbone checkpoint', c); break
    print('features not found -> backbone (reuse checkpoint if present, else train) + extraction')

In [ ]:
# Cell 9: backbone fine-tuning + feature extraction (runs only when Cell 8 found no features)
if not HAVE_FEATS:
    # full fine-tuning of the backbone on the backbone pool (no test data anywhere in this cell)
    def build_backbone(pretrained):
        try:
            return timm.create_model(CFG['backbone'], pretrained=pretrained, num_classes=K, drop_rate=0.2)
        except Exception as e:
            if not FAST_DEV_RUN:
                raise
            print('pretrained weights unavailable in smoke test -> random init:', type(e).__name__)
            return timm.create_model(CFG['backbone'], pretrained=False, num_classes=K, drop_rate=0.2)

    @torch.no_grad()
    def run_backbone(model, df, noise=None, bs=128):
        # returns pooled features (pre-logits) and softmax probabilities of the end-to-end model
        model.eval(); feats, probs = [], []
        for x, _ in make_loader(df, bs=bs, noise=noise):
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                f = model.forward_head(model.forward_features(x), pre_logits=True)
                logit = model.get_classifier()(f)
            feats.append(f.float().cpu()); probs.append(torch.softmax(logit.float(), 1).cpu())
        return torch.cat(feats).numpy(), torch.cat(probs).numpy()

    CKPT = f"{OUT}/backbone_{CFG['backbone'].replace('.', '_')}.pt"
    backbone = build_backbone(CFG['pretrained']).to(DEVICE).to(memory_format=torch.channels_last)
    bb_hist = []

    if CFG['reuse_backbone'] and os.path.exists(CKPT):
        backbone.load_state_dict(torch.load(CKPT, map_location=DEVICE)); print('loaded', CKPT)
    else:
        seed_everything(GLOBAL_SEED)
        tr, va = POOLS['backbone'], POOLS['backbone_val']
        cls_w = 1.0 / tr.label.value_counts().reindex(range(K)).values
        sampler = WeightedRandomSampler(torch.tensor(cls_w[tr.label.values], dtype=torch.double),
                                        num_samples=min(len(tr), CFG['bb_samples_per_epoch']),
                                        replacement=True, generator=torch.Generator().manual_seed(GLOBAL_SEED))
        tr_loader = make_loader(tr, train=True, bs=CFG['bb_bs'], sampler=sampler)
        head_p = list(backbone.get_classifier().parameters()); head_ids = {id(p) for p in head_p}
        body_p = [p for p in backbone.parameters() if id(p) not in head_ids]
        opt = torch.optim.AdamW([{'params': body_p, 'lr': CFG['bb_lr_body']},
                                 {'params': head_p, 'lr': CFG['bb_lr_head']}], weight_decay=CFG['bb_wd'])
        total = CFG['bb_epochs'] * len(tr_loader); warm = max(1, len(tr_loader) // 2)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
        scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP)
        crit = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
        best, bad = -1.0, 0
        for ep in range(CFG['bb_epochs']):
            backbone.train(); t0 = time.time(); tl, n = 0.0, 0
            for x, y in tr_loader:
                x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                    loss = crit(backbone(x), y)
                opt.zero_grad(set_to_none=True); scaler_amp.scale(loss).backward()
                scaler_amp.step(opt); scaler_amp.update(); sched.step()
                tl += loss.item() * len(y); n += len(y)
            _, pv = run_backbone(backbone, va)
            m = metrics(va.label.values, pv)
            bb_hist.append(dict(epoch=ep + 1, train_loss=tl / n, **{f'val_{k}': v for k, v in m.items()}))
            print(f"BB ep {ep+1:02d} loss {tl/n:.4f} | backbone-val acc {m['acc']:.4f} bacc {m['bacc']:.4f} "
                  f"f1 {m['f1']:.4f} | {time.time()-t0:.0f}s")
            if m['f1'] > best:
                best, bad = m['f1'], 0; torch.save(backbone.state_dict(), CKPT)
            else:
                bad += 1
                if bad >= CFG['bb_patience']:
                    print('early stop'); break
        backbone.load_state_dict(torch.load(CKPT, map_location=DEVICE))
        pd.DataFrame(bb_hist).to_csv(f'{OUT}/backbone_history.csv', index=False)
        h = pd.DataFrame(bb_hist)
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].plot(h.epoch, h.train_loss, 'o-'); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
        for k in ['val_acc', 'val_bacc', 'val_f1']:
            ax[1].plot(h.epoch, h[k], 'o-', label=k)
        ax[1].set_title('backbone-val (patient-disjoint)'); ax[1].legend(); ax[1].set_xlabel('epoch')
        plt.tight_layout(); plt.savefig(f'{OUT}/backbone_training.png', dpi=150); plt.show()
    print('best backbone-val macro-F1 checkpoint ready')
    for p in backbone.parameters():
        p.requires_grad = False
    for name in ['head', 'locked_test', 'official_test']:
        f, pr = run_backbone(backbone, POOLS[name])
        FEAT[name] = dict(X=f, bb_prob=pr, y=POOLS[name].label.values, g=POOLS[name].patient.values)
        print(name, f.shape)
    np.savez_compressed(f'{OUT}/features.npz', **{f'{k}_{a}': v[a] for k, v in FEAT.items() for a in ['X', 'bb_prob', 'y']})
FEAT_DIM = FEAT['head']['X'].shape[1]
print('feature dim', FEAT_DIM)

## Exact batched simulator for the quantum layer

Each of the `C` parallel circuits: `RY` angle embedding on 5 wires → `L` StronglyEntanglingLayers (PennyLane convention: `Rot(φ,θ,ω)=RZ(ω)RY(θ)RZ(φ)` on each wire, then a CNOT ring with range `(l mod (n-1))+1`) → ⟨Y⟩ on every wire.

* **Statevector path** (training, noiseless): exact, differentiable, batched over samples × circuits.
* **Density-matrix path** (evaluation): single-qubit depolarising channel with probability *p* after every gate (after a CNOT, on both its wires).
* **Finite shots**: each ⟨Y_w⟩ is replaced by its *S*-shot estimate, 2·Binomial(S, (1+⟨Y_w⟩)/2)/S − 1 (marginally exact).

The next cell checks the simulator against PennyLane before anything is trained with it.

In [ ]:
# Cell 10: torch quantum simulator
CDT = torch.complex64

def _ry(th):
    c, s = torch.cos(th / 2), torch.sin(th / 2)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(CDT)

def _rz(ph):
    e = torch.exp(-0.5j * ph.to(CDT)); z = torch.zeros_like(e)
    return torch.stack([torch.stack([e, z], -1), torch.stack([z, e.conj()], -1)], -2)

def _rot(w):                       # w (...,3) -> Rot(phi,theta,omega) = RZ(omega) RY(theta) RZ(phi)
    return _rz(w[..., 2]) @ _ry(w[..., 1]) @ _rz(w[..., 0])

PX = torch.tensor([[0, 1], [1, 0]], dtype=CDT)
PY = torch.tensor([[0, -1j], [1j, 0]], dtype=CDT)
PZ = torch.tensor([[1, 0], [0, -1]], dtype=CDT)

def _apply_1q(t, U, axis):
    # t: (B, C, 2, ..., 2); applies U on `axis`: out_i = sum_j U_ij t_j.  U: (2,2) | (C,2,2) | (B,C,2,2)
    B, C = t.shape[:2]
    t = t.movedim(axis, -1); shp = t.shape
    t2 = t.reshape(B, C, -1, 2)
    if U.dim() == 2:
        out = t2 @ U.transpose(0, 1)
    elif U.dim() == 3:
        out = torch.einsum('cij,bckj->bcki', U, t2)
    else:
        out = torch.einsum('bcij,bckj->bcki', U, t2)
    return out.reshape(shp).movedim(-1, axis)

def _apply_cnot(t, ac, at):
    t = t.movedim((ac, at), (-2, -1))
    t = torch.stack([t[..., 0, :], t[..., 1, :].flip(-1)], dim=-2)
    return t.movedim((-2, -1), (ac, at))

def _depol(rho, p, w, n):
    out = (1 - p) * rho
    for P in (PX.to(rho.device), PY.to(rho.device), PZ.to(rho.device)):
        out = out + (p / 3) * _apply_1q(_apply_1q(rho, P, 2 + w), P.conj(), 2 + n + w)
    return out

def qsim(angles, weights, depol=0.0, shots=None, gen=None):
    # angles (B,C,n) real; weights (C,L,n,3) real -> <Y_w> (B,C,n)
    B, C, n = angles.shape; L = weights.shape[1]; dev = angles.device
    ranges = [(l % (n - 1)) + 1 for l in range(L)] if n > 1 else [0] * L
    emb = _ry(angles)                      # (B,C,n,2,2)
    rot = _rot(weights)                    # (C,L,n,2,2)
    if depol == 0.0:
        psi = torch.zeros(B, C, 2 ** n, dtype=CDT, device=dev); psi[:, :, 0] = 1
        psi = psi.reshape(B, C, *([2] * n))
        for w in range(n):
            psi = _apply_1q(psi, emb[:, :, w], 2 + w)
        for l in range(L):
            for w in range(n):
                psi = _apply_1q(psi, rot[:, l, w], 2 + w)
            for w in range(n):
                psi = _apply_cnot(psi, 2 + w, 2 + (w + ranges[l]) % n)
        ev = []
        for w in range(n):
            a0, a1 = psi.select(2 + w, 0), psi.select(2 + w, 1)
            ev.append(2 * (a0.conj() * a1).imag.sum(dim=tuple(range(2, 1 + n))))   # <Y> = 2 Im(a0* a1)
        ev = torch.stack(ev, -1)
    else:
        rho = torch.zeros(B, C, 2 ** n, 2 ** n, dtype=CDT, device=dev); rho[:, :, 0, 0] = 1
        rho = rho.reshape(B, C, *([2] * (2 * n)))
        def U_(r, U, w):
            return _apply_1q(_apply_1q(r, U, 2 + w), U.conj(), 2 + n + w)
        for w in range(n):
            rho = _depol(U_(rho, emb[:, :, w], w), depol, w, n)
        for l in range(L):
            for w in range(n):
                rho = _depol(U_(rho, rot[:, l, w], w), depol, w, n)
            for w in range(n):
                c, t = w, (w + ranges[l]) % n
                rho = _apply_cnot(_apply_cnot(rho, 2 + c, 2 + t), 2 + n + c, 2 + n + t)
                rho = _depol(_depol(rho, depol, c, n), depol, t, n)
        ev = []
        for w in range(n):
            yr = _apply_1q(rho, PY.to(dev), 2 + w).reshape(B, C, 2 ** n, 2 ** n)
            ev.append(torch.diagonal(yr, dim1=-2, dim2=-1).sum(-1).real)
        ev = torch.stack(ev, -1)
    if shots:
        p_plus = ((1 + ev) / 2).clamp(0, 1).float()
        k = torch.binomial(torch.full_like(p_plus, float(shots)), p_plus, generator=gen)
        ev = 2 * k / shots - 1
    return ev.float()
print('simulator defined')

In [ ]:
# Cell 11: verify the simulator against PennyLane (outputs, gradients, depolarising density matrix)
import pennylane as qml
n, L = CFG['q_qubits'], CFG['q_layers']
ranges = [(l % (n - 1)) + 1 for l in range(L)]
dev_sv = qml.device('default.qubit', wires=n)
dev_dm = qml.device('default.mixed', wires=n)

@qml.qnode(dev_sv, interface='torch', diff_method='backprop')
def ref_sv(x, w):
    qml.AngleEmbedding(x, wires=range(n), rotation='Y')
    qml.StronglyEntanglingLayers(w, wires=range(n))
    return [qml.expval(qml.PauliY(i)) for i in range(n)]

@qml.qnode(dev_dm, interface='torch')
def ref_dm(x, w, p):
    for i in range(n):
        qml.RY(x[i], wires=i); qml.DepolarizingChannel(p, wires=i)
    for l in range(L):
        for i in range(n):
            qml.Rot(w[l, i, 0], w[l, i, 1], w[l, i, 2], wires=i); qml.DepolarizingChannel(p, wires=i)
        for i in range(n):
            j = (i + ranges[l]) % n
            qml.CNOT(wires=[i, j]); qml.DepolarizingChannel(p, wires=i); qml.DepolarizingChannel(p, wires=j)
    return [qml.expval(qml.PauliY(i)) for i in range(n)]

g = torch.Generator().manual_seed(0); max_err = dict(sv=0.0, grad=0.0, dm=0.0)
for trial in range(5):
    x = (torch.rand(n, generator=g) * 2 - 1) * math.pi
    w = torch.rand(L, n, 3, generator=g) * 2 * math.pi
    # forward + gradient, noiseless
    w_ref = w.clone().double().requires_grad_(True); w_ours = w.clone().requires_grad_(True)
    coef = torch.randn(n, generator=g)
    r = torch.stack(list(ref_sv(x.double(), w_ref))).float(); (r * coef).sum().backward()
    o = qsim(x.view(1, 1, n), w_ours.view(1, L, n, 3))[0, 0]; (o * coef).sum().backward()
    max_err['sv'] = max(max_err['sv'], (r.detach() - o.detach()).abs().max().item())
    max_err['grad'] = max(max_err['grad'], (w_ref.grad.float() - w_ours.grad).abs().max().item())
    # depolarising
    p = [1e-3, 1e-2, 5e-2, 0.1, 0.2][trial]
    rd = torch.stack(list(ref_dm(x.double(), w.double(), p))).float()
    od = qsim(x.view(1, 1, n), w.view(1, L, n, 3), depol=p)[0, 0]
    max_err['dm'] = max(max_err['dm'], (rd - od).abs().max().item())
print('max |torch - PennyLane| :', {k: f'{v:.2e}' for k, v in max_err.items()})
assert all(v < 1e-4 for v in max_err.values()), 'simulator does not match PennyLane'
print('Simulator matches PennyLane (statevector, gradients, depolarising density matrix).')

## A. Data re-uploading VQC

Per circuit (5 qubits), per layer *l*: `RY(s_{l,w} · a_w)` on every wire → `Rot(φ,θ,ω)` on every wire → CNOT ring with range `(l mod 4)+1`; measure ⟨Y⟩ on every wire.
`a = π·tanh(z)` is the same bounded encoding as the 2-layer `hybrid_vqc` head, and `s` are trainable input scales.
Re-uploading *L* times raises the accessible Fourier frequencies of each input from 1 to *L* (Schuld, Sweke & Meyer, *PRA* 103, 032430, 2021), so this is the more expressive circuit family.

| Head | Middle block | Middle params |
|---|---|---|
| `reupload_L3` | 3 re-uploading layers | 4 × (3·15 + 3·5) = 240 |
| `classical_deep_2` | 4 × [Linear(5,5)→tanh]×2 | 4 × 2 × 30 = 240 |
| `reupload_L6` | 6 re-uploading layers | 480 |
| `classical_deep_4` | 4 × [Linear(5,5)→tanh]×4 | 480 |

The baseline heads of notebook 01 (`hybrid_vqc`, `classical_matched`, `bottleneck_only`, `logreg`) are retrained here on the same folds and seeds, so every comparison is paired.

In [ ]:
# Cell 12: re-uploading simulator + check against PennyLane
def qsim_reupload(angles, scales, weights, shots=None, gen=None):
    # angles (B,C,n); scales (C,L,n); weights (C,L,n,3) -> <Y_w> (B,C,n)
    B, C, n = angles.shape; L = weights.shape[1]
    ranges = [(l % (n - 1)) + 1 for l in range(L)]
    rot = _rot(weights)
    psi = torch.zeros(B, C, 2 ** n, dtype=CDT, device=angles.device); psi[:, :, 0] = 1
    psi = psi.reshape(B, C, *([2] * n))
    for l in range(L):
        emb = _ry(angles * scales[:, l].unsqueeze(0))          # (B,C,n,2,2)
        for w in range(n):
            psi = _apply_1q(psi, emb[:, :, w], 2 + w)
        for w in range(n):
            psi = _apply_1q(psi, rot[:, l, w], 2 + w)
        for w in range(n):
            psi = _apply_cnot(psi, 2 + w, 2 + (w + ranges[l]) % n)
    ev = torch.stack([2 * (psi.select(2 + w, 0).conj() * psi.select(2 + w, 1)).imag.sum(dim=tuple(range(2, 1 + n)))
                      for w in range(n)], -1).float()
    if shots:
        k = torch.binomial(torch.full_like(ev, float(shots)), ((1 + ev) / 2).clamp(0, 1), generator=gen)
        ev = 2 * k / shots - 1
    return ev

def _check_reupload(L):
    n = CFG['q_qubits']; rg = [(l % (n - 1)) + 1 for l in range(L)]
    @qml.qnode(qml.device('default.qubit', wires=n), interface='torch', diff_method='backprop')
    def ref(x, s, w):
        for l in range(L):
            for i in range(n):
                qml.RY(s[l, i] * x[i], wires=i)
            for i in range(n):
                qml.Rot(w[l, i, 0], w[l, i, 1], w[l, i, 2], wires=i)
            for i in range(n):
                qml.CNOT(wires=[i, (i + rg[l]) % n])
        return [qml.expval(qml.PauliY(i)) for i in range(n)]
    g = torch.Generator().manual_seed(L); err = grad_err = 0.0
    for _ in range(3):
        x = (torch.rand(n, generator=g) * 2 - 1) * math.pi
        s = torch.rand(L, n, generator=g) * 2
        w = torch.rand(L, n, 3, generator=g) * 2 * math.pi
        coef = torch.randn(n, generator=g)
        wr = w.double().clone().requires_grad_(True); sr = s.double().clone().requires_grad_(True)
        r = torch.stack(list(ref(x.double(), sr, wr))).float(); (r * coef).sum().backward()
        wo = w.clone().requires_grad_(True); so = s.clone().requires_grad_(True)
        o = qsim_reupload(x.view(1, 1, n), so.view(1, L, n), wo.view(1, L, n, 3))[0, 0]; (o * coef).sum().backward()
        err = max(err, (r.detach() - o.detach()).abs().max().item())
        grad_err = max(grad_err, (wr.grad.float() - wo.grad).abs().max().item(), (sr.grad.float() - so.grad).abs().max().item())
    return err, grad_err

for L in CFG['reupload_layers']:
    e, ge = _check_reupload(L)
    print(f're-uploading L={L}: max |torch - PennyLane| output {e:.2e}, gradient {ge:.2e}')
    assert e < 1e-4 and ge < 1e-4
print('re-uploading simulator matches PennyLane')

In [ ]:
# Cell 13: head family (baseline VQC, re-uploading VQCs and their parameter-matched classical controls)
QC, QN, QL = CFG['q_circuits'], CFG['q_qubits'], CFG['q_layers']
QW = QC * QN

class Head(nn.Module):
    def __init__(self, kind, in_dim, seed):
        super().__init__()
        self.kind = kind
        g = torch.Generator().manual_seed(10_000 + seed)
        self.drop_in = nn.Dropout(CFG['head_in_dropout'])
        self.enc = nn.Linear(in_dim, QW)
        self.drop_mid = nn.Dropout(CFG['q_dropout'])
        self.out = nn.Linear(QW, K)
        bound = 1 / math.sqrt(QN)
        if kind == 'hybrid_vqc':
            self.qw = nn.Parameter(torch.rand(QC, QL, QN, 3, generator=g) * 2 * math.pi)
        elif kind == 'hybrid_vqc_frozen':
            self.register_buffer('qw', torch.rand(QC, QL, QN, 3, generator=g) * 2 * math.pi)
        elif kind == 'classical_matched':
            self.cw = nn.Parameter((torch.rand(QC, 1, QN, QN, generator=g) * 2 - 1) * bound)
            self.cb = nn.Parameter((torch.rand(QC, 1, QN, generator=g) * 2 - 1) * bound)
        elif kind.startswith('reupload_L'):
            L = int(kind.split('L')[-1])
            self.qw = nn.Parameter(torch.rand(QC, L, QN, 3, generator=g) * 2 * math.pi)
            self.qs = nn.Parameter(torch.ones(QC, L, QN))
        elif kind.startswith('classical_deep_'):
            D = int(kind.split('_')[-1])
            self.cw = nn.Parameter((torch.rand(QC, D, QN, QN, generator=g) * 2 - 1) * bound)
            self.cb = nn.Parameter((torch.rand(QC, D, QN, generator=g) * 2 - 1) * bound)
        elif kind != 'bottleneck_only':
            raise ValueError(kind)

    def middle(self, z, depol=0.0, shots=None, gen=None):
        B = z.shape[0]
        if self.kind == 'bottleneck_only':
            return torch.tanh(z)
        a = (math.pi * torch.tanh(z)).view(B, QC, QN)
        if self.kind.startswith('hybrid'):
            return qsim(a, self.qw, depol=depol, shots=shots, gen=gen).reshape(B, QW)
        if self.kind.startswith('reupload'):
            return qsim_reupload(a, self.qs, self.qw, shots=shots, gen=gen).reshape(B, QW)
        h = a
        for d in range(self.cw.shape[1]):
            h = torch.tanh(torch.einsum('cij,bcj->bci', self.cw[:, d], h) + self.cb[:, d])
        return h.reshape(B, QW)

    def forward(self, x, **qkw):
        return self.out(self.drop_mid(self.middle(self.enc(self.drop_in(x)), **qkw)))

PAIRS_A = [('hybrid_vqc', 'classical_matched')]
for L in CFG['reupload_layers']:
    PAIRS_A.append((f'reupload_L{L}', f'classical_deep_{2 * L // 3}'))
TORCH_HEADS = ['bottleneck_only'] + [h for p in PAIRS_A for h in p][::-1]
TORCH_HEADS = list(dict.fromkeys(TORCH_HEADS))
ALL_HEADS = ['logreg'] + TORCH_HEADS

rows = []
for kd in TORCH_HEADS:
    h = Head(kd, FEAT_DIM, 0)
    rows.append(dict(head=kd, trainable_total=sum(p.numel() for p in h.parameters() if p.requires_grad),
                     middle=sum(p.numel() for n_, p in h.named_parameters() if n_ in ('qw', 'qs', 'cw', 'cb'))))
PARAMS = pd.DataFrame(rows); print(PARAMS.to_string(index=False))
for q, c in PAIRS_A:
    mq, mc = (PARAMS.set_index('head').loc[x, 'middle'] for x in (q, c))
    assert mq == mc, f'parameter mismatch {q}={mq} vs {c}={mc}'
print('pairs (quantum, matched classical):', PAIRS_A)
PARAMS.to_csv(f'{OUT}/benchmark_param_counts.csv', index=False)

In [ ]:
# Cell 14: head training (identical recipe for every torch head) and prediction
def train_head(kind, Xtr, ytr, Xva, yva, seed, epochs=None, min_steps=None):
    seed_everything(seed)
    model = Head(kind, Xtr.shape[1], seed).to(DEVICE)
    decay = [p for n_, p in model.named_parameters() if n_ in ('enc.weight', 'out.weight')]
    other = [p for n_, p in model.named_parameters() if p.requires_grad and n_ not in ('enc.weight', 'out.weight')]
    opt = torch.optim.Adam([{'params': decay, 'weight_decay': CFG['head_wd']},
                            {'params': other, 'weight_decay': 0.0}], lr=CFG['head_lr'])
    counts = np.bincount(ytr, minlength=K).astype(float)
    cw = torch.tensor(len(ytr) / (K * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=CFG['label_smoothing'])
    N = len(ytr); bs = min(CFG['head_bs'], N); spe = math.ceil(N / bs)
    total = max((epochs or CFG['head_epochs']) * spe, min_steps or CFG['head_min_steps'])
    eval_every = max(spe, 25)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
    Xt = torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE); yt = torch.as_tensor(ytr, device=DEVICE)
    gperm = torch.Generator(device='cpu').manual_seed(seed)
    best, best_state, bad, step = -1.0, None, 0, 0
    perm = torch.randperm(N, generator=gperm); pos = 0
    while step < total:
        model.train()
        if pos + bs > N:
            perm = torch.randperm(N, generator=gperm); pos = 0
        idx = perm[pos:pos + bs].to(DEVICE); pos += bs
        loss = crit(model(Xt[idx]), yt[idx])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); step += 1
        if step % eval_every == 0 or step == total:
            f1 = metrics(yva, predict_head(model, Xva))['f1']
            if f1 > best + 1e-6:
                best, bad, best_state = f1, 0, copy.deepcopy(model.state_dict())
            else:
                bad += 1
                if bad >= CFG['head_patience']:
                    break
    model.load_state_dict(best_state); model.eval()
    return model, dict(steps=step, best_val_f1=best)

@torch.no_grad()
def predict_head(model, X, bs=4096, depol=0.0, shots=None, seed=0):
    model.eval()
    if depol > 0:
        bs = 256
    gen = torch.Generator(device=DEVICE).manual_seed(seed) if shots else None
    out = []
    for i in range(0, len(X), bs):
        xb = torch.as_tensor(X[i:i + bs], dtype=torch.float32, device=DEVICE)
        out.append(torch.softmax(model(xb, depol=depol, shots=shots, gen=gen), 1).cpu())
    return torch.cat(out).numpy()

def fit_logreg(Xtr, ytr, Xva, yva):
    best = None
    for C in [0.01, 0.1, 1.0]:
        m = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
        f1 = metrics(yva, m.predict_proba(Xva))['f1']
        if best is None or f1 > best[0]:
            best = (f1, m, C)
    return best[1], dict(C=best[2], best_val_f1=best[0])
print('ok')

In [ ]:
# Cell 15: 5-fold patient-grouped CV x seeds
if RUN_BENCHMARK:
    # identical folds and seeds to notebook 01
    Xh, yh, gh = FEAT['head']['X'], FEAT['head']['y'], FEAT['head']['g']
    FOLDS = list(StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=GLOBAL_SEED).split(Xh, yh, gh))
    MODELS, SCALERS, CV_ROWS = {}, {}, []
    for fold, (tri, vai) in enumerate(FOLDS):
        sc = StandardScaler().fit(Xh[tri]); SCALERS[fold] = sc
        Xtr, Xva, ytr, yva = sc.transform(Xh[tri]), sc.transform(Xh[vai]), yh[tri], yh[vai]
        m, _ = fit_logreg(Xtr, ytr, Xva, yva); MODELS[('logreg', fold, 0)] = m
        for s in CFG['head_seeds']:
            CV_ROWS.append(dict(fold=fold, seed=s, head='logreg', n_train=len(tri), n_val=len(vai), **metrics(yva, m.predict_proba(Xva))))
        for s in CFG['head_seeds']:
            for kd in TORCH_HEADS:
                t0 = time.time()
                m, info = train_head(kd, Xtr, ytr, Xva, yva, seed=1000 * fold + s)
                MODELS[(kd, fold, s)] = m
                CV_ROWS.append(dict(fold=fold, seed=s, head=kd, n_train=len(tri), n_val=len(vai), secs=time.time() - t0,
                                    **metrics(yva, predict_head(m, Xva))))
        now = pd.DataFrame(CV_ROWS); now = now[now.fold == fold].groupby('head').f1.mean()
        print(f'fold {fold}: ' + ' | '.join(f'{h} {now[h]:.4f}' for h in ALL_HEADS))
    CV = pd.DataFrame(CV_ROWS); CV.to_csv(f'{OUT}/benchmark_cv_heads.csv', index=False)

## B. Quantum-kernel SVMs

Features: PCA of the frozen 512-d features to *n* = 8 or 12 dims (fit on fold-train), standardised. Then:

* **IQP fidelity kernel** (Havlíček et al., *Nature* 567, 209, 2019): |ψ(x)⟩ = D(x) H⊗ⁿ D(x) H⊗ⁿ |0⟩, with D(x) = Π RZ(2b·x_i) Π_{i<j} ZZ(2b²·x_i x_j), and k(x,x') = |⟨ψ(x)|ψ(x')⟩|².
* **Projected quantum kernel** (Huang et al., 2021): k = exp(−γ Σ_k ‖r_k(x) − r_k(x')‖²), with r_k the single-qubit Bloch vectors of |ψ(x)⟩.
* **Controls:** RBF-SVM on the same *n*-d PCA features (the matched comparison); RBF-SVM and logistic regression on the *n*-d inputs and on the full 512-d features (reference ceiling).

Bandwidth *b*, γ and C are chosen on a patient-grouped inner split of fold-train, never on fold-val.
SVMs train on a class-stratified subsample of `kern_train_n` images per fold, the same subsample for every kernel. The states are simulated exactly and checked against PennyLane.

In [ ]:
# Cell 16: IQP states, kernels, check vs PennyLane, concentration diagnostic  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    from scipy.linalg import hadamard
    from sklearn.svm import SVC
    from sklearn.decomposition import PCA
    from sklearn.model_selection import GroupShuffleSplit, train_test_split

    _WHT = {}
    def iqp_states(X, b, bs=2048):
        # X (N,n) standardised -> complex statevectors (N, 2^n), wire 0 = most significant bit (PennyLane order)
        N, n = X.shape; d = 2 ** n
        if n not in _WHT:
            _WHT[n] = (torch.tensor(hadamard(d), dtype=torch.complex64, device=DEVICE) / math.sqrt(d),
                       torch.tensor(1 - 2 * ((np.arange(d)[:, None] >> (n - 1 - np.arange(n))[None]) & 1),
                                    dtype=torch.float64, device=DEVICE))   # spins s in {+1,-1}, (d, n)
        W, Sp = _WHT[n]; out = []
        for i in range(0, N, bs):
            x = torch.as_tensor(X[i:i + bs], dtype=torch.float64, device=DEVICE)
            lin = x @ Sp.T                                              # sum_i x_i s_i
            phase = b * lin + 0.5 * b * b * (lin ** 2 - (x ** 2).sum(1, keepdim=True))   # + b^2 sum_{i<j} x_i x_j s_i s_j
            D = torch.exp(-1j * phase).to(torch.complex64)
            psi = D / math.sqrt(d)                                      # D H|0>
            psi = (psi @ W) * D                                         # D H D H|0>   (W symmetric)
            out.append(psi)
        return torch.cat(out)

    def fidelity_kernel(Pa, Pb, bs=4096):
        return torch.cat([(Pa[i:i + bs] @ Pb.conj().T).abs().pow(2) for i in range(0, len(Pa), bs)]).double().cpu().numpy()

    def bloch(P, n):
        t = P.reshape(len(P), *([2] * n)); r = []
        for k in range(n):
            a0, a1 = t.select(1 + k, 0).reshape(len(P), -1), t.select(1 + k, 1).reshape(len(P), -1)
            c = (a0.conj() * a1).sum(1)
            r += [2 * c.real, 2 * c.imag, (a0.abs() ** 2 - a1.abs() ** 2).sum(1)]
        return torch.stack(r, 1).double().cpu().numpy()

    def rbf_kernel(A, B, gamma):
        d = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None] - 2 * A @ B.T
        return np.exp(-gamma * np.maximum(d, 0))

    # ---- verify against PennyLane
    nchk, bchk = 4, 0.37
    @qml.qnode(qml.device('default.qubit', wires=nchk))
    def iqp_ref(x):
        for _ in range(2):
            for i in range(nchk):
                qml.Hadamard(wires=i)
            for i in range(nchk):
                qml.RZ(2 * bchk * x[i], wires=i)
            for i in range(nchk):
                for j in range(i + 1, nchk):
                    qml.IsingZZ(2 * bchk * bchk * x[i] * x[j], wires=[i, j])
        return qml.state()
    rs = np.random.RandomState(0); xa, xb = rs.randn(2, nchk)
    Pa, Pb = iqp_states(np.stack([xa, xb]), bchk).cpu().numpy()
    ra, rb = np.asarray(iqp_ref(xa)), np.asarray(iqp_ref(xb))
    err_k = abs(abs(np.vdot(Pa, Pb)) ** 2 - abs(np.vdot(ra, rb)) ** 2)
    err_s = np.abs(Pa - ra * np.exp(1j * np.angle(np.vdot(ra, Pa)))).max()
    @qml.qnode(qml.device('default.qubit', wires=nchk))
    def bloch_ref(x):
        for _ in range(2):
            for i in range(nchk):
                qml.Hadamard(wires=i)
            for i in range(nchk):
                qml.RZ(2 * bchk * x[i], wires=i)
            for i in range(nchk):
                for j in range(i + 1, nchk):
                    qml.IsingZZ(2 * bchk * bchk * x[i] * x[j], wires=[i, j])
        return [qml.expval(o(i)) for i in range(nchk) for o in (qml.PauliX, qml.PauliY, qml.PauliZ)]
    err_b = np.abs(bloch(torch.tensor(Pa[None], device=DEVICE), nchk)[0] - np.array([float(v) for v in bloch_ref(xa)])).max()
    print(f'IQP vs PennyLane: state (up to global phase) {err_s:.1e}, kernel {err_k:.1e}, Bloch vectors {err_b:.1e}')
    assert err_s < 1e-4 and err_k < 1e-5 and err_b < 1e-4

    # ---- exponential-concentration diagnostic (Thanasilp et al., Nat. Commun. 2024)
    sub = np.random.RandomState(1).choice(len(Xh), min(500, len(Xh)), replace=False)
    CONC = []
    for n in CFG['kern_qubits']:
        Z = StandardScaler().fit_transform(PCA(n, random_state=0).fit_transform(Xh[sub]))
        for b in CFG['kern_bandwidths'] + [1.6, 3.2]:
            Kq = fidelity_kernel(*(2 * [iqp_states(Z, b)]))
            off = Kq[~np.eye(len(Kq), dtype=bool)]
            CONC.append(dict(qubits=n, bandwidth=b, offdiag_mean=off.mean(), offdiag_std=off.std(), uniform_level=2.0 ** -n))
    CONC = pd.DataFrame(CONC); print(CONC.round(5).to_string(index=False)); CONC.to_csv(f'{OUT}/benchmark_kernel_concentration.csv', index=False)
    print('A kernel whose off-diagonal mean/std approaches 2^-n carries no information (concentration).')

In [ ]:
# Cell 17: kernel-SVM CV (hyper-parameters chosen on an inner patient-grouped split of fold-train)  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    def svc_fit_eval(Ktr, ytr, Kva, yva, C):
        m = SVC(C=C, kernel='precomputed', class_weight='balanced', decision_function_shape='ovr').fit(Ktr, ytr)
        return m, metrics(yva, m.decision_function(Kva))

    def kernel_builders(n, Ztr_full):
        # returns {model_name: (grid list, function(params, A, B) -> K)}; inputs are standardised PCA features
        def fid(p, A, B): return fidelity_kernel(iqp_states(A, p['b']), iqp_states(B, p['b']))
        def proj(p, A, B):
            return rbf_kernel(bloch(iqp_states(A, p['b']), n), bloch(iqp_states(B, p['b']), n), p['g'] / (3 * n))
        def rbf(p, A, B): return rbf_kernel(A, B, p['g'] / A.shape[1])
        bw, gs, ps = CFG['kern_bandwidths'], CFG['rbf_gamma_scale'], CFG['proj_gamma_scale']
        return {f'qsvm_fidelity_q{n}': ([dict(b=b) for b in bw], fid),
                f'qsvm_projected_q{n}': ([dict(b=b, g=g) for b in bw for g in ps], proj),
                f'rbf_svm_pca{n}': ([dict(g=g) for g in gs], rbf)}

    KCV, KPRED, KSEL = [], defaultdict(dict), []
    for fold, (tri, vai) in enumerate(FOLDS):
        sub = tri if len(tri) <= CFG['kern_train_n'] else train_test_split(
            tri, train_size=CFG['kern_train_n'], stratify=yh[tri], random_state=500 + fold)[0]
        gss = GroupShuffleSplit(1, test_size=CFG['kern_inner_val'], random_state=fold)
        itr, iva = next(gss.split(sub, yh[sub], gh[sub])); itr, iva = sub[itr], sub[iva]
        sc512 = StandardScaler().fit(Xh[sub])
        # reference ceilings on full features (same subsample)
        for name, kfun, grid in [('rbf_svm_full', lambda p, A, B: rbf_kernel(A, B, p['g'] / A.shape[1]),
                                  [dict(g=g) for g in CFG['rbf_gamma_scale']])]:
            A, Bv = sc512.transform(Xh[itr]), sc512.transform(Xh[iva])
            best = max(((p, C, svc_fit_eval(kfun(p, A, A), yh[itr], kfun(p, Bv, A), yh[iva], C)[1]['f1'])
                        for p in grid for C in CFG['kern_C']), key=lambda t: t[2])
            S = sc512.transform(Xh[sub]); m, mv = svc_fit_eval(kfun(best[0], S, S), yh[sub], kfun(best[0], sc512.transform(Xh[vai]), S), yh[vai], best[1])
            KCV.append(dict(fold=fold, head=name, n_train=len(sub), n_val=len(vai), **mv))
            KSEL.append(dict(fold=fold, head=name, params=best[0], C=best[1]))
            for split in ['locked_test', 'official_test']:
                KPRED[name][(split, fold)] = m.decision_function(kfun(best[0], sc512.transform(FEAT[split]['X']), S))
        for n in CFG['kern_qubits']:
            pca = PCA(n, random_state=fold).fit(Xh[sub]); zs = StandardScaler().fit(pca.transform(Xh[sub]))
            Z = lambda X: zs.transform(pca.transform(X))
            Zi, Zv, Zs = Z(Xh[itr]), Z(Xh[iva]), Z(Xh[sub])
            # linear reference on the same n-d inputs
            lr, _ = fit_logreg(Zs, yh[sub], Z(Xh[vai]), yh[vai])
            KCV.append(dict(fold=fold, head=f'logreg_pca{n}', n_train=len(sub), n_val=len(vai), **metrics(yh[vai], lr.predict_proba(Z(Xh[vai])))))
            for split in ['locked_test', 'official_test']:
                KPRED[f'logreg_pca{n}'][(split, fold)] = lr.predict_proba(Z(FEAT[split]['X']))
            for name, (grid, kfun) in kernel_builders(n, Zs).items():
                t0 = time.time(); scores = []
                for p in grid:
                    Ki, Kv = kfun(p, Zi, Zi), kfun(p, Zv, Zi)
                    for C in CFG['kern_C']:
                        scores.append((p, C, svc_fit_eval(Ki, yh[itr], Kv, yh[iva], C)[1]['f1']))
                p, C, inner = max(scores, key=lambda t: t[2])
                m, mv = svc_fit_eval(kfun(p, Zs, Zs), yh[sub], kfun(p, Z(Xh[vai]), Zs), yh[vai], C)
                KCV.append(dict(fold=fold, head=name, n_train=len(sub), n_val=len(vai), secs=time.time() - t0, **mv))
                KSEL.append(dict(fold=fold, head=name, params=p, C=C, inner_f1=inner))
                for split in ['locked_test', 'official_test']:
                    KPRED[name][(split, fold)] = m.decision_function(kfun(p, Z(FEAT[split]['X']), Zs))
        now = pd.DataFrame(KCV); now = now[now.fold == fold].set_index('head').f1
        print(f'fold {fold}: ' + ' | '.join(f'{h} {v:.4f}' for h, v in now.items()))

    KCV = pd.DataFrame(KCV); KCV.to_csv(f'{OUT}/benchmark_cv_kernels.csv', index=False)
    KSEL = pd.DataFrame(KSEL); KSEL.to_csv(f'{OUT}/benchmark_kernel_selected_params.csv', index=False)
    print('\nselected hyper-parameters per fold:'); print(KSEL.to_string(index=False))
    PAIRS_B = []
    for n in CFG['kern_qubits']:
        PAIRS_B += [(f'qsvm_fidelity_q{n}', f'rbf_svm_pca{n}'), (f'qsvm_projected_q{n}', f'rbf_svm_pca{n}')]

## C. Geometric difference and model complexity

Following Huang et al. (2021), with both kernels normalised to tr K = N:

* **Geometric difference** g(K_C‖K_Q) = √‖ √K_Q · √K_C (K_C+λI)⁻² √K_C · √K_Q ‖_∞.
  If g ≪ √N, the classical kernel can match the quantum kernel's generalisation for **every** labelling, so no quantum advantage is possible for this data representation, whatever the labels.
* **Model complexity** on the real labels: s_K = ‖√K (K+λI)⁻¹ Y‖² / K_classes (one-vs-rest ±1 targets). Generalisation error scales as √(s/N). If s_C ≤ s_Q, the classical kernel is predicted to generalise at least as well **on this task**.

K_Q: fidelity and projected IQP kernels at each fold-0-selected bandwidth (plus the whole bandwidth grid for g). K_C: RBF at the fold-0-selected γ, and the linear kernel.

In [ ]:
# Cell 18: geometric difference on a stratified head-pool subsample  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    def _eig(K):
        w, Vv = np.linalg.eigh((K + K.T) / 2); return np.clip(w, 0, None), Vv
    def _norm(K): return K * (len(K) / np.trace(K))
    def geo_diff(KC, KQ, lam):
        wC, VC = _eig(KC); wQ, VQ = _eig(KQ)
        sQ = (VQ * np.sqrt(wQ)) @ VQ.T
        mid = (VC * (wC / (wC + lam) ** 2)) @ VC.T          # sqrt(KC) (KC+lam)^-2 sqrt(KC)
        return float(np.sqrt(np.linalg.eigvalsh(sQ @ mid @ sQ).max()))
    def complexity(Kk, Y, lam):
        w, Vv = _eig(Kk); P = Vv.T @ Y
        return float(((w / (w + lam) ** 2)[:, None] * P ** 2).sum() / Y.shape[1])

    idx = train_test_split(np.arange(len(Xh)), train_size=min(CFG['geo_n'], len(Xh) - K), stratify=yh, random_state=7)[0]
    N = len(idx); Y = np.where(np.eye(K)[yh[idx]] > 0, 1.0, -1.0)
    sel0 = KSEL[KSEL.fold == 0].set_index('head')
    GEO = []
    for n in CFG['kern_qubits']:
        Z = StandardScaler().fit_transform(PCA(n, random_state=0).fit_transform(Xh[idx]))
        g_rbf = sel0.loc[f'rbf_svm_pca{n}', 'params']['g']
        KCs = {'rbf(selected)': _norm(rbf_kernel(Z, Z, g_rbf / n)), 'linear': _norm(Z @ Z.T + 1e-9 * np.eye(N))}
        pp = sel0.loc[f'qsvm_projected_q{n}', 'params']
        KQs = {f'fidelity b={b}': _norm(fidelity_kernel(*(2 * [iqp_states(Z, b)])) ) for b in CFG['kern_bandwidths']}
        KQs[f"projected b={pp['b']} (selected)"] = _norm(rbf_kernel(*(2 * [bloch(iqp_states(Z, pp['b']), n)]), pp['g'] / (3 * n)))
        sel_b = sel0.loc[f'qsvm_fidelity_q{n}', 'params']['b']
        for lam in CFG['geo_lambdas']:
            sC = {c: complexity(Kc, Y, lam) for c, Kc in KCs.items()}
            for qn, Kq in KQs.items():
                sq = complexity(Kq, Y, lam)
                for cn, Kc in KCs.items():
                    GEO.append(dict(qubits=n, lam=lam, K_Q=qn + (' (selected)' if qn == f'fidelity b={sel_b}' else ''),
                                    K_C=cn, g_CQ=geo_diff(Kc, Kq, lam), sqrtN=math.sqrt(N), s_C=sC[cn], s_Q=sq, N=N))
        print(f'{n} qubits done')
    GEO = pd.DataFrame(GEO); GEO.to_csv(f'{OUT}/benchmark_geometric_difference.csv', index=False)
    pd.set_option('display.width', 200)
    print(GEO.round(3).to_string(index=False))

    fig, axs = plt.subplots(1, len(CFG['kern_qubits']), figsize=(6.5 * len(CFG['kern_qubits']), 4.5), squeeze=False)
    for ax, n in zip(axs[0], CFG['kern_qubits']):
        d = GEO[(GEO.qubits == n) & (GEO.K_C == 'rbf(selected)')]
        for qn, dd in d.groupby('K_Q'):
            ax.plot(dd.lam, dd.g_CQ, 'o-', label=qn)
        ax.axhline(math.sqrt(N), color='k', ls='--', label='sqrt(N)')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('regularisation lambda'); ax.set_ylabel('g(K_C || K_Q)')
        ax.set_title(f'{n} qubits, K_C = RBF (selected gamma), N = {N}'); ax.legend(fontsize=7); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{OUT}/benchmark_geometric_difference.png', dpi=150); plt.show()

In [ ]:
# Cell 19: CV summary + corrected resampled t-tests for the pre-declared pairs  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    ALL_CV = pd.concat([CV, KCV.assign(seed=0)], ignore_index=True)
    tab = ALL_CV.groupby('head')[['acc', 'bacc', 'f1']].agg(['mean', 'std'])
    print('CV (fold-val), mean +- std:'); print((tab * 100).round(2).to_string())

    def corrected_ttest(d, n_train, n_test):
        d = np.asarray(d, float); J = len(d); v = d.var(ddof=1)
        if J < 2 or v == 0: return 0.0, 1.0
        t = d.mean() / math.sqrt((1 / J + n_test / n_train) * v)
        return t, 2 * stats.t.sf(abs(t), J - 1)
    def holm(p):
        p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
        for r, i in enumerate(o):
            run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
        return adj

    PAIRS = PAIRS_A + PAIRS_B
    rows = []
    for q, c in PAIRS:
        a = ALL_CV[ALL_CV['head'] == q].set_index(['fold', 'seed']).f1
        b = ALL_CV[ALL_CV['head'] == c].set_index(['fold', 'seed']).f1
        d = (a - b).dropna().values
        nt = ALL_CV[ALL_CV['head'] == q].n_train.mean(); nv = ALL_CV[ALL_CV['head'] == q].n_val.mean()
        t, p = corrected_ttest(d, nt, nv)
        rows.append(dict(quantum=q, classical=c, delta_f1_pp=100 * d.mean(), folds_quantum_better=f'{(d > 0).sum()}/{len(d)}', t=t, p=p))
    CVT = pd.DataFrame(rows); CVT['p_holm'] = holm(CVT.p)
    print('\nPre-declared paired comparisons (CV macro-F1):'); print(CVT.round(4).to_string(index=False))
    CVT.to_csv(f'{OUT}/benchmark_cv_tests.csv', index=False)

    order = ['logreg', 'bottleneck_only'] + [h for p in PAIRS_A for h in p[::-1]] + \
            ['rbf_svm_full'] + [h for n in CFG['kern_qubits'] for h in (f'logreg_pca{n}', f'rbf_svm_pca{n}', f'qsvm_fidelity_q{n}', f'qsvm_projected_q{n}')]
    order = [h for h in order if h in ALL_CV['head'].unique()]
    plt.figure(figsize=(max(10, 0.8 * len(order)), 4.8))
    quantum = lambda h: h.startswith(('hybrid', 'reupload', 'qsvm'))
    pal = {h: ('#c0504d' if quantum(h) else '#4f81bd') for h in order}
    sns.boxplot(data=ALL_CV, x='head', y='f1', order=order, hue='head', hue_order=order, palette=pal, legend=False)
    plt.xticks(rotation=40, ha='right'); plt.ylabel('fold-val macro-F1'); plt.xlabel('')
    plt.title('CV: quantum (red) vs classical (blue), identical folds'); plt.tight_layout()
    plt.savefig(f'{OUT}/benchmark_cv_boxplot.png', dpi=150); plt.show()

In [ ]:
# Cell 20: LOCKED + OFFICIAL TEST for the pre-declared pairs (run once)  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    assert UNLOCK_TEST, 'Test sets locked.'
    def cluster_boot_idx(groups, B, seed):
        r = np.random.RandomState(seed); ug, inv = np.unique(groups, return_inverse=True)
        mem = [np.where(inv == k)[0] for k in range(len(ug))]
        for _ in range(B):
            yield np.concatenate([mem[k] for k in r.randint(0, len(ug), len(ug))])
    def test_scores(h, split):
        if h in KPRED:
            return np.mean([v for (s, f), v in KPRED[h].items() if s == split], 0)
        X = FEAT[split]['X']; out = []
        for (hh, fold, s), m in MODELS.items():
            if hh == h:
                Xs = SCALERS[fold].transform(X)
                out.append(m.predict_proba(Xs) if h == 'logreg' else predict_head(m, Xs))
        return np.mean(out, 0)

    heads = sorted({h for p in PAIRS for h in p} | {'logreg', 'rbf_svm_full'})
    RES, TT = [], []
    for split in ['locked_test', 'official_test']:
        y, g = FEAT[split]['y'], FEAT[split]['g']
        P = {h: test_scores(h, split) for h in heads}
        boots = list(cluster_boot_idx(g, CFG['n_boot'], 123))
        for h in heads:
            pr = P[h].argmax(1); bf = [f1_score(y[i], pr[i], average='macro') for i in boots]
            RES.append(dict(split=split, head=h, **metrics(y, P[h]), f1_lo=np.percentile(bf, 2.5), f1_hi=np.percentile(bf, 97.5)))
        for q, c in PAIRS:
            a, b = P[q].argmax(1), P[c].argmax(1)
            d = np.array([f1_score(y[i], a[i], average='macro') - f1_score(y[i], b[i], average='macro') for i in boots])
            n01, n10 = int(((a == y) & (b != y)).sum()), int(((a != y) & (b == y)).sum())
            TT.append(dict(split=split, quantum=q, classical=c,
                           delta_f1_pp=100 * (f1_score(y, a, average='macro') - f1_score(y, b, average='macro')),
                           ci_lo_pp=100 * np.percentile(d, 2.5), ci_hi_pp=100 * np.percentile(d, 97.5),
                           quantum_only_correct=n01, classical_only_correct=n10,
                           p_mcnemar=stats.binomtest(min(n01, n10), n01 + n10, .5).pvalue if n01 + n10 else 1.0,
                           p_mcnemar_clustered=clustered_mcnemar(a == y, b == y, g),   # patients contribute several images
                           p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
    RES, TT = pd.DataFrame(RES), pd.DataFrame(TT)
    for s in TT.split.unique():
        m = TT.split == s
        TT.loc[m, 'p_mcnemar_holm'] = holm(TT.loc[m, 'p_mcnemar']); TT.loc[m, 'p_boot_holm'] = holm(TT.loc[m, 'p_boot'])
        TT.loc[m, 'p_mcnemar_clustered_holm'] = holm(TT.loc[m, 'p_mcnemar_clustered'])
    RES.to_csv(f'{OUT}/benchmark_test_metrics.csv', index=False); TT.to_csv(f'{OUT}/benchmark_test_comparisons.csv', index=False)
    for s in ['locked_test', 'official_test']:
        print(f'\n=== {s} ==='); r = RES[RES.split == s].set_index('head')
        print((r[['acc', 'bacc', 'f1', 'f1_lo', 'f1_hi']] * 100).round(2).to_string())
        print(TT[TT.split == s].drop(columns='split').round(4).to_string(index=False))

In [ ]:
# Cell 21: verdicts  [skipped unless RUN_BENCHMARK]
if RUN_BENCHMARK:
    lines = ['| Quantum model | Matched classical | CV delta macro-F1 (pp) | CV p (Holm) | Locked-test delta (pp) [95% CI] | McNemar p (Holm) | Verdict |',
             '|---|---|---|---|---|---|---|']
    for _, r in CVT.iterrows():
        t = TT[(TT.split == 'locked_test') & (TT.quantum == r.quantum)].iloc[0]
        if t.ci_lo_pp > 0 and r.p_holm < 0.05:   v = 'quantum better'
        elif t.ci_hi_pp < 0 and r.p_holm < 0.05: v = 'classical better'
        elif r.p_holm < 0.05:                    v = ('classical' if r.delta_f1_pp < 0 else 'quantum') + ' better in CV only'
        else:                                    v = 'no detectable difference'
        lines.append(f'| {r.quantum} | {r.classical} | {r.delta_f1_pp:+.2f} ({r.folds_quantum_better}) | {r.p_holm:.3g} | '
                     f'{t.delta_f1_pp:+.2f} [{t.ci_lo_pp:+.2f}, {t.ci_hi_pp:+.2f}] | {t.p_mcnemar_holm:.3g} | {v} |')
    table = '\n'.join(lines); print(table); open(f'{OUT}/benchmark_verdicts.md', 'w').write(table + '\n')

    for n in CFG['kern_qubits']:
        for lam in CFG['geo_lambdas']:
            dd = GEO[(GEO.qubits == n) & (GEO.lam == lam) & GEO.K_Q.str.contains('selected')]
            for _, r in dd.iterrows():
                print(f"{n}q lambda={lam:g} {r.K_Q:28s} vs {r.K_C:13s}: g_CQ = {r.g_CQ:6.2f} (g/sqrt(N) = {r.g_CQ / r.sqrtN:.2f}); "
                      f"s_C = {r.s_C:8.1f}, s_Q = {r.s_Q:8.1f}")
    print('Reading (Huang et al. 2021): g/sqrt(N) well below 1 -> the classical kernel can match the quantum kernel for any labels; '
          'compare s_C and s_Q only between kernels that also fit the training data.')
    print('\nAll outputs in', OUT)

## IBM hardware: trained hybrid VQC on a real QPU

One trained `hybrid_vqc` head (fold 0, seed 0; identical to the CV model with that fold and seed) and its matched classical twin.
For each image, the 4 parallel 5-qubit sub-circuits are packed into **one 20-qubit circuit**: RY embedding → 2 × StronglyEntanglingLayers → S†·H → measure, so every wire is read out in the Y basis.
The encoder `Linear(512→20)` and the output layer `Linear(20→4)` run classically.

Steps (Cell 4 `IBM_STEP`):
0. Order: `check` → `test` → `main` (unmitigated, Sampler) → `mit_test` → `mitigated` (Estimator with TREX readout mitigation, gate twirling and DD, same images and same physical qubits).
1. `'check'`: verifies the Qiskit circuit against the PyTorch model, connects to the backend, transpiles, and records gate counts and calibration errors. **No QPU time.**
2. `'test'`: 20 images × 256 shots. Expect a clearly positive correlation between hardware and exact ⟨Y⟩; check the usage on the IBM dashboard.
3. `'main'`: all images × 1024 shots in one job. Smaller shot counts (16–256) come from subsampling the measured shots.
4. `'resume'` + `IBM_JOB_ID`: fetch a finished job after a session restart, without using more QPU time.

Secrets needed (Add-ons → Secrets, enabled for this notebook): `IBM_QUANTUM_TOKEN` and, optionally, `IBM_QUANTUM_INSTANCE` (the instance CRN).

In [ ]:
# Cell 22: trained head, balanced locked-test subset, circuit inputs (no Qiskit in this process)
_saved = f'{OUT}/hw_heads_fold0_seed0.pt'
if os.path.exists(_saved) and os.path.exists(f'{OUT}/hw_images.csv') and IBM_STEP not in ('check', 'test'):
    # reload exactly what was submitted (retraining could differ slightly because of GPU non-determinism)
    ck = torch.load(_saved, map_location=DEVICE, weights_only=False)
    vqc, cls = Head('hybrid_vqc', FEAT_DIM, 0).to(DEVICE), Head('classical_matched', FEAT_DIM, 0).to(DEVICE)
    vqc.load_state_dict(ck['vqc']); cls.load_state_dict(ck['cls'])
    SC = StandardScaler(); SC.mean_, SC.scale_ = ck['scaler_mean'], ck['scaler_scale']
    SC.var_, SC.n_features_in_ = SC.scale_ ** 2, len(SC.mean_)
    IBM_N_IMAGES = len(pd.read_csv(f'{OUT}/hw_images.csv'))
    _resume_sel = pd.read_csv(f'{OUT}/hw_images.csv').locked_index.values
    print('reloaded the head and image list used on the QPU from', OUT)
elif 'MODELS' in globals() and ('hybrid_vqc', 0, 0) in MODELS:
    vqc, cls, SC = MODELS[('hybrid_vqc', 0, 0)], MODELS[('classical_matched', 0, 0)], SCALERS[0]
    print('using the CV models from cell 15')
else:
    Xh, yh, gh = FEAT['head']['X'], FEAT['head']['y'], FEAT['head']['g']
    FOLDS = list(StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=GLOBAL_SEED).split(Xh, yh, gh))
    tri, vai = FOLDS[0]; SC = StandardScaler().fit(Xh[tri])
    vqc, iq = train_head('hybrid_vqc', SC.transform(Xh[tri]), yh[tri], SC.transform(Xh[vai]), yh[vai], seed=0)
    cls, ic = train_head('classical_matched', SC.transform(Xh[tri]), yh[tri], SC.transform(Xh[vai]), yh[vai], seed=0)
    print(f"trained fold-0/seed-0 heads: val macro-F1 hybrid_vqc {iq['best_val_f1']:.4f}, classical_matched {ic['best_val_f1']:.4f}")
vqc.eval(); cls.eval()

XL, yL, gL = FEAT['locked_test']['X'], FEAT['locked_test']['y'], FEAT['locked_test']['g']
rs = np.random.RandomState(2026); per = IBM_N_IMAGES // K
SEL = _resume_sel if IBM_STEP not in ('check', 'test') and '_resume_sel' in globals() else np.sort(np.concatenate([rs.choice(np.where(yL == c)[0], min(per, (yL == c).sum()), replace=False) for c in range(K)]))
yS, gS = yL[SEL], gL[SEL]
with torch.no_grad():
    XS = torch.tensor(SC.transform(XL[SEL]), dtype=torch.float32, device=DEVICE)
    ANG = (math.pi * torch.tanh(vqc.enc(XS))).cpu().numpy().astype(np.float64)                       # (N, 20)
    EV_EXACT = qsim(torch.tensor(ANG, dtype=torch.float32, device=DEVICE).view(-1, QC, QN), vqc.qw).reshape(-1, QW).cpu().numpy()
    P_CLS = torch.softmax(cls(XS), 1).cpu().numpy()
QW_NP = vqc.qw.detach().cpu().numpy().astype(np.float64)
np.savez(f'{OUT}/hw_inputs.npz', ANG=ANG, QW_NP=QW_NP, EV_EXACT=EV_EXACT)
if IBM_STEP in ('check', 'test'):
  torch.save({'vqc': vqc.state_dict(), 'cls': cls.state_dict(), 'scaler_mean': SC.mean_, 'scaler_scale': SC.scale_},
           f'{OUT}/hw_heads_fold0_seed0.pt')
if IBM_STEP in ('check', 'test'):
  pd.DataFrame({'locked_index': SEL, 'file': POOLS['locked_test'].path.iloc[SEL].map(os.path.basename).values,
              'patient': gS, 'label': [CLASSES[i] for i in yS]}).to_csv(f'{OUT}/hw_images.csv', index=False)

def head_from_ev(ev):
    with torch.no_grad():
        return torch.softmax(vqc.out(torch.tensor(ev, dtype=torch.float32, device=DEVICE)), -1).cpu().numpy()
P_EXACT = head_from_ev(EV_EXACT)
print(f'{len(SEL)} images / {len(np.unique(gS))} patients', dict(zip(CLASSES, np.bincount(yS, minlength=K))))
print('exact simulator:', {k: round(v, 4) for k, v in metrics(yS, P_EXACT).items()},
      '| classical matched:', {k: round(v, 4) for k, v in metrics(yS, P_CLS).items()})

In [ ]:
# Cell 23: write the Qiskit runner to disk (it runs in its own process)
IBM_SCRIPT = os.path.join(OUT, 'ibm_run.py')
open(IBM_SCRIPT, 'w').write(r'''"""Runs the trained hybrid VQC on IBM hardware in its own process (avoids TLS clashes with torch/sklearn).

Modes
  check          verify the circuit, connect, transpile, record calibration (no QPU time)
  test           Sampler, first --test-n images x --test-shots shots, dynamical decoupling only
  main           Sampler, all images x --main-shots shots, dynamical decoupling only (unmitigated reference)
  mit_test       Estimator, first --test-n images: TREX readout mitigation + gate twirling + DD
  mitigated      Estimator, all (or --mit-n) images: TREX + gate twirling + DD   [--zne adds zero-noise extrapolation]
  resume         fetch any earlier job by --job-id (its mode is looked up in the saved job files)

Reads  RESULTS/hw_inputs.npz (ANG, QW_NP, EV_EXACT)
Writes RESULTS/hw_bits_<mode>.npy (Sampler) or RESULTS/hw_ev_<mode>.npy (Estimator), hw_info_<mode>.json, ibm_job_<mode>.json"""
import os, sys, json, glob, argparse
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

ap = argparse.ArgumentParser()
ap.add_argument('mode', choices=['check', 'test', 'main', 'mit_test', 'mitigated', 'resume'])
ap.add_argument('--job-id', default='')
ap.add_argument('--backend', default='ibm_kingston')
ap.add_argument('--fake', action='store_true', help='use local FakeTorino instead of IBM (no QPU time)')
ap.add_argument('--results', default='/kaggle/working/results' if os.path.isdir('/kaggle/working') else 'results')
ap.add_argument('--test-n', type=int, default=20)
ap.add_argument('--test-shots', type=int, default=256)
ap.add_argument('--main-shots', type=int, default=1024)
ap.add_argument('--mit-n', type=int, default=0, help='images for the mitigated run (0 = all)')
ap.add_argument('--mit-shots', type=int, default=1024)
ap.add_argument('--zne', action='store_true', help='also apply zero-noise extrapolation (resilience level 2; ~3x QPU time)')
args = ap.parse_args()
R = args.results

z = np.load(f'{R}/hw_inputs.npz')
ANG, QW_NP, EV_EXACT = z['ANG'], z['QW_NP'], z['EV_EXACT']
C, L, n, _ = QW_NP.shape
W = C * n
RANGES = [(l % (n - 1)) + 1 for l in range(L)]


def build(measure):
    # C parallel n-qubit sub-circuits, PennyLane conventions; measure=True adds the Y-basis readout
    a = ParameterVector('a', W)
    qc = QuantumCircuit(W, W) if measure else QuantumCircuit(W)
    for c in range(C):
        q = [c * n + i for i in range(n)]
        for i in range(n):
            qc.ry(a[c * n + i], q[i])                                   # AngleEmbedding(rotation='Y')
        for l in range(L):
            for i in range(n):
                phi, theta, omega = QW_NP[c, l, i]
                qc.rz(phi, q[i]); qc.ry(theta, q[i]); qc.rz(omega, q[i])  # Rot(phi, theta, omega)
            for i in range(n):
                qc.cx(q[i], q[(i + RANGES[l]) % n])                     # StronglyEntanglingLayers ring
    if measure:
        for j in range(W):
            qc.sdg(j); qc.h(j)                                          # |+i> -> |0>
        qc.measure(range(W), range(W))
    return qc, a


qc, a = build(measure=True)

# ---- noiseless check against the PyTorch model (no QPU time)
for i in range(3):
    b = qc.assign_parameters(dict(zip(a, ANG[i])))
    bits = StatevectorSampler(seed=i).run([b], shots=20000).result()[0].data.c.to_bool_array(order='little')
    err = np.abs((1 - 2 * bits.mean(0)) - EV_EXACT[i]).max()
    print(f'check image {i}: max |sampled - exact| = {err:.3f}')
    if err > 0.05:
        sys.exit('CIRCUIT DOES NOT MATCH THE TRAINED MODEL -- stop, do not submit')
print('circuit verified against the PyTorch model')

# ---- which kind of job is this?
job_meta = {}
if args.mode == 'resume':
    for f in glob.glob(f'{R}/ibm_job_*.json'):
        m = json.load(open(f))
        if m.get('job_id') == args.job_id:
            job_meta = m
    if not job_meta:
        print('job id not found in saved job files -> assuming a Sampler "main" job')
        job_meta = dict(mode='main', primitive='sampler')
primitive = job_meta.get('primitive') or ('estimator' if args.mode in ('mit_test', 'mitigated') else 'sampler')
tag = job_meta.get('mode', args.mode)

# ---- backend
if args.fake:
    from qiskit_ibm_runtime.fake_provider import FakeTorino
    backend, service = FakeTorino(), None
else:
    from qiskit_ibm_runtime import QiskitRuntimeService
    token = os.environ.get('IBM_QUANTUM_TOKEN', '')
    if not token:
        sys.exit('IBM_QUANTUM_TOKEN not set in the environment')
    service = QiskitRuntimeService(channel='ibm_quantum_platform', token=token,
                                   instance=os.environ.get('IBM_QUANTUM_INSTANCE') or None)
    print('QPUs available:', [x.name for x in service.backends(simulator=False)])
    backend = service.backend(args.backend)
    st = backend.status()
    print(f'{backend.name}: {backend.num_qubits} qubits | operational {st.operational} | queue {st.pending_jobs}')
    if not st.operational and args.mode not in ('check', 'resume'):
        sys.exit('backend not operational right now -- try later')

# ---- transpile once; mitigated runs reuse the physical qubits of the unmitigated main run
init_layout = None
_main_info = next((f for f in [f'{R}/hw_info_main.json', f'{R}/hw_info.json'] if os.path.exists(f)), None)
if primitive == 'estimator' and _main_info:
    prev = json.load(open(_main_info))
    if prev.get('backend') == backend.name and len(prev.get('physical_qubits', [])) == W:
        init_layout = prev['physical_qubits']
        print('using the physical qubits of the main run:', init_layout)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=42, initial_layout=init_layout)
circ, a_c = (qc, a) if primitive == 'sampler' else build(measure=False)
isa = pm.run(circ)
two_q = {k: v for k, v in isa.count_ops().items() if k in ('cz', 'ecr', 'cx')}
layout = list(isa.layout.final_index_layout())
gname = next(iter(two_q), None); tgt = backend.target; e2, ero = [], []
for ins in isa.data:
    if ins.operation.name == gname:
        p = tgt[gname].get(tuple(isa.find_bit(x).index for x in ins.qubits))
        if p is not None and p.error is not None:
            e2.append(p.error)
for qb in layout:
    p = tgt['measure'].get((qb,)) if 'measure' in tgt else None
    if p is not None and p.error is not None:
        ero.append(p.error)
info = dict(backend=backend.name, primitive=primitive, logical_cnots=C * L * n, two_qubit_gates=two_q,
            depth=isa.depth(), depth_2q=isa.depth(lambda ins: ins.operation.num_qubits == 2), physical_qubits=layout,
            median_2q_error=float(np.median(e2)) if e2 else None,
            median_readout_error=float(np.median(ero)) if ero else None)
print('transpiled:', info)
json.dump(info, open(f'{R}/hw_info_{tag}.json', 'w'), indent=2, default=str)   # one record per step
if tag == 'main':
    json.dump(info, open(f'{R}/hw_info.json', 'w'), indent=2, default=str)  # the unmitigated main run is the reference
VALS = ANG[:, [list(a_c).index(p) for p in isa.parameters]]

if args.mode == 'check':
    print(f'check only, nothing submitted. Main run: {len(ANG)} images x {args.main_shots} shots '
          f'= {len(ANG) * args.main_shots:,} shots.')
    sys.exit(0)

# ---- submit or resume
if args.mode == 'resume':
    if not args.job_id or service is None:
        sys.exit('resume needs --job-id and a real IBM connection')
    job = service.job(args.job_id)
    print(f'resuming {primitive} job {args.job_id} (mode {tag})')
else:
    if args.mode == 'test':
        idx, shots = np.arange(min(args.test_n, len(VALS))), args.test_shots
    elif args.mode == 'main':
        idx, shots = np.arange(len(VALS)), args.main_shots
    elif args.mode == 'mit_test':
        idx, shots = np.arange(min(args.test_n, len(VALS))), args.mit_shots
    else:
        idx, shots = np.arange(args.mit_n if args.mit_n else len(VALS)), args.mit_shots
    if primitive == 'sampler':
        from qiskit_ibm_runtime import SamplerV2 as Sampler
        sampler = Sampler(mode=backend)
        sampler.options.default_shots = shots
        if not args.fake:
            sampler.options.dynamical_decoupling.enable = True
        job = sampler.run([(isa, VALS[idx])])
        settings = dict(dynamical_decoupling=not args.fake)
    else:
        from qiskit_ibm_runtime import EstimatorV2 as Estimator
        est = Estimator(mode=backend)
        est.options.default_shots = shots
        if not args.fake:
            est.options.resilience_level = 2 if args.zne else 1          # 1 = TREX readout mitigation
            est.options.resilience.measure_mitigation = True              # TREX
            est.options.twirling.enable_gates = True                      # Pauli twirling of 2-qubit gates
            est.options.twirling.enable_measure = True                    # measurement twirling (part of TREX)
            est.options.dynamical_decoupling.enable = True
            est.options.dynamical_decoupling.sequence_type = 'XpXm'
            if args.zne:
                est.options.resilience.zne_mitigation = True
        obs = [[SparsePauliOp.from_sparse_list([('Y', [j], 1.0)], num_qubits=W).apply_layout(isa.layout)
                for j in range(W)]]                                       # nested list -> observables shape (1, W)
        job = est.run([(isa, obs, VALS[idx][:, None, :])])                # values (N, 1, P) -> evs (N, W)
        settings = dict(TREX=not args.fake, gate_twirling=not args.fake, dynamical_decoupling=not args.fake,
                        zne=bool(args.zne and not args.fake))
    json.dump(dict(job_id=job.job_id(), mode=args.mode, primitive=primitive, shots=shots, n=int(len(idx)),
                   backend=backend.name, settings=settings),
              open(f'{R}/ibm_job_{args.mode}.json', 'w'), indent=2)
    print(f'JOB ID: {job.job_id()}  ({primitive}, {len(idx)} images x {shots} shots, {settings}) '
          f'-> saved to {R}/ibm_job_{args.mode}.json', flush=True)

res = job.result()                                                      # waits in the queue
if primitive == 'sampler':
    BITS = res[0].data.c.to_bool_array(order='little')                  # (N, shots, W); index j = logical qubit j
    np.save(f'{R}/hw_bits_{tag}.npy', BITS)
    ev = 1 - 2 * BITS.mean(1)
    print(f'saved {R}/hw_bits_{tag}.npy  shape {BITS.shape}')
else:
    ev = np.asarray(res[0].data.evs, dtype=float).reshape(-1, W)        # mitigated <Y_j> per image
    std = np.asarray(res[0].data.stds, dtype=float).reshape(-1, W)
    np.save(f'{R}/hw_ev_{tag}.npy', ev); np.save(f'{R}/hw_ev_std_{tag}.npy', std)
    print(f'saved {R}/hw_ev_{tag}.npy  shape {ev.shape}')
ref = EV_EXACT[:len(ev)]
print(f'hardware vs exact <Y>: Pearson r = {np.corrcoef(ev.ravel(), ref.ravel())[0, 1]:.3f}, '
      f'MAE = {np.abs(ev - ref).mean():.3f}, attenuation slope = {(ev * ref).sum() / (ref ** 2).sum():.3f}')
try:
    print('QPU usage (s):', job.usage())
except Exception:
    pass
''')
print('wrote', IBM_SCRIPT)

In [ ]:
# Cell 24: install Qiskit (not imported here), pass the IBM key to the subprocess, run the chosen step
if IBM_STEP == 'none':
    print("IBM_STEP = 'none' -> nothing submitted to IBM (analysis only)")
else:
    get_ipython().system('pip install -q qiskit qiskit-ibm-runtime qiskit-aer')
    if not IBM_FAKE and not os.environ.get('IBM_QUANTUM_TOKEN'):      # Colab cell 0 already set it from Colab secrets
        try:
            from kaggle_secrets import UserSecretsClient
            _sec = UserSecretsClient()
            os.environ['IBM_QUANTUM_TOKEN'] = _sec.get_secret('IBM_QUANTUM_TOKEN')
            try:
                os.environ['IBM_QUANTUM_INSTANCE'] = _sec.get_secret('IBM_QUANTUM_INSTANCE')
            except Exception:
                print('no IBM_QUANTUM_INSTANCE secret -> searching all your instances')
        except Exception as e:
            print('could not read Kaggle secrets:', e)
    if not IBM_FAKE:
        assert os.environ.get('IBM_QUANTUM_TOKEN'), 'Add the IBM_QUANTUM_TOKEN secret (Kaggle: Add-ons -> Secrets; Colab: key icon) and allow access.'

    cmd = f'python {IBM_SCRIPT} {IBM_STEP} --backend {IBM_BACKEND} --results {OUT}'
    if IBM_STEP == 'resume':
        assert IBM_JOB_ID, 'set IBM_JOB_ID in Cell 4'
        cmd += f' --job-id {IBM_JOB_ID}'
    if IBM_STEP == 'mitigated' and IBM_MIT_N:
        cmd += f' --mit-n {IBM_MIT_N}'
    if IBM_STEP in ('mit_test', 'mitigated') and IBM_ZNE:
        cmd += ' --zne'
    if IBM_FAKE:
        cmd += ' --fake'
    print('$', cmd, flush=True)
    import subprocess
    _p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                          env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONWARNINGS': 'ignore::DeprecationWarning'})
    for _line in _p.stdout:
        print(_line, end='', flush=True)
    _rc = _p.wait()
    if _rc != 0:
        raise RuntimeError(f'ibm_run.py exited with code {_rc} -- read the messages above')
    print('\nNext:', {'check': "if 'circuit verified' was printed, set IBM_STEP = 'test' and re-run Cell 4 and this cell",
                      'test': "if Pearson r is clearly positive, set IBM_STEP = 'main' (check your remaining minutes first)",
                      'main': "run IBM-4; then (quota permitting) set IBM_STEP = 'mit_test'",
                      'mit_test': "if the mitigated Pearson r / attenuation look sensible, set IBM_STEP = 'mitigated'",
                      'mitigated': 'run IBM-4 and IBM-5', 'resume': 'run IBM-4 and IBM-5'}[IBM_STEP])

In [ ]:
# Cell 25: analysis -- hardware vs exact vs shot noise vs classical, on the same images
_bits = f'{OUT}/hw_bits_test.npy' if IBM_STEP == 'test' else f'{OUT}/hw_bits_main.npy'
if not os.path.exists(_bits):
    print('no hardware results yet:', _bits)
else:
    BITS = np.load(_bits); n_hw = BITS.shape[0]
    y_, g_, PE, PC, EVX = yS[:n_hw], gS[:n_hw], P_EXACT[:n_hw], P_CLS[:n_hw], EV_EXACT[:n_hw]
    EV_HW = 1 - 2 * BITS.mean(1)
    r_ = np.random.RandomState(99); ug, inv = np.unique(g_, return_inverse=True)
    mem = [np.where(inv == k)[0] for k in range(len(ug))]
    BOOTS = [np.concatenate([mem[k] for k in r_.randint(0, len(ug), len(ug))]) for _ in range(CFG['n_boot'])]
    pe = PE.argmax(1); rng = np.random.RandomState(7); rows = []

    def f1ci(P):
        pr = P.argmax(1); v = [f1_score(y_[i], pr[i], average='macro') for i in BOOTS]
        return np.percentile(v, 2.5), np.percentile(v, 97.5)

    for name, P in [('exact simulator', PE), ('classical matched', PC), ('IBM hardware', head_from_ev(EV_HW))]:
        lo, hi = f1ci(P); m = metrics(y_, P)
        rows.append(dict(condition=name, shots=BITS.shape[1] if name == 'IBM hardware' else np.nan, f1=m['f1'], f1_sd=0.0,
                         f1_lo=lo, f1_hi=hi, acc=m['acc'], agree_exact=(P.argmax(1) == pe).mean()))
    for S in [s for s in [16, 64, 256, 1024] if s < BITS.shape[1]]:
        for name in ['IBM hardware', 'shot noise only (sim)']:
            Ps = []
            for _ in range(20):
                if name == 'IBM hardware':
                    pick = np.stack([rng.choice(BITS.shape[1], S, replace=False) for _ in range(n_hw)])
                    ev = 1 - 2 * np.take_along_axis(BITS, pick[:, :, None], 1).mean(1)
                else:
                    ev = 2 * rng.binomial(S, (1 + EVX) / 2) / S - 1
                Ps.append(head_from_ev(ev))
            f = [metrics(y_, P)['f1'] for P in Ps]
            rows.append(dict(condition=name, shots=S, f1=np.mean(f), f1_sd=np.std(f), f1_lo=np.nan, f1_hi=np.nan,
                             acc=np.mean([metrics(y_, P)['acc'] for P in Ps]),
                             agree_exact=np.mean([(P.argmax(1) == pe).mean() for P in Ps])))
    full_s = BITS.shape[1]
    ev = 2 * rng.binomial(full_s, (1 + EVX) / 2) / full_s - 1
    rows.append(dict(condition='shot noise only (sim)', shots=full_s, f1=metrics(y_, head_from_ev(ev))['f1'], f1_sd=0.0,
                     f1_lo=np.nan, f1_hi=np.nan, acc=metrics(y_, head_from_ev(ev))['acc'], agree_exact=(head_from_ev(ev).argmax(1) == pe).mean()))
    HWR = pd.DataFrame(rows).sort_values(['condition', 'shots']); HWR.to_csv(f'{OUT}/hw_results.csv', index=False)
    print((HWR.set_index(['condition', 'shots']) * 100).round(2).to_string())

    P_HW = head_from_ev(EV_HW); a_ = P_HW.argmax(1)
    n01, n10 = int(((a_ == y_) & (pe != y_)).sum()), int(((a_ != y_) & (pe == y_)).sum())
    p_mc = clustered_mcnemar(a_ == y_, pe == y_, g_)          # clustered by patient
    d = np.array([f1_score(y_[i], a_[i], average='macro') - f1_score(y_[i], pe[i], average='macro') for i in BOOTS])
    slope = float((EV_HW * EVX).sum() / (EVX ** 2).sum()); corr = float(np.corrcoef(EV_HW.ravel(), EVX.ravel())[0, 1])
    with torch.no_grad():                         # where the hardware sits on the simulated depolarising curve
        A_t = torch.tensor(ANG[:min(64, n_hw)], dtype=torch.float32, device=DEVICE).view(-1, QC, QN); E0 = EVX[:min(64, n_hw)]
        grid = [0.0, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05]
        att = [float((qsim(A_t, vqc.qw, depol=p).reshape(-1, QW).cpu().numpy() * E0).sum() / (E0 ** 2).sum()) for p in grid]
    p_eff = float(np.interp(slope, att[::-1], grid[::-1]))
    info = json.load(open(next(f for f in [f'{OUT}/hw_info_main.json', f'{OUT}/hw_info.json'] if os.path.exists(f))))
    SUMMARY = dict(**info, n_images=int(n_hw), patients=int(len(ug)), shots=int(full_s),
                   delta_f1_hw_minus_exact_pp=100 * (metrics(y_, P_HW)['f1'] - metrics(y_, PE)['f1']),
                   delta_ci_pp=[100 * np.percentile(d, 2.5), 100 * np.percentile(d, 97.5)],
                   mcnemar_clustered=dict(hw_only_correct=n01, exact_only_correct=n10, p=p_mc), p_boot=boot_p(d),
                   attenuation_slope=slope, pearson_r=corr, effective_uniform_depol_p=p_eff)
    print(json.dumps(SUMMARY, indent=2, default=str))
    json.dump(SUMMARY, open(f'{OUT}/hw_summary.json', 'w'), indent=2, default=str)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    for cond, mk, col in [('IBM hardware', 'o-', '#c0504d'), ('shot noise only (sim)', 's--', '#4f81bd')]:
        dd = HWR[(HWR.condition == cond) & HWR.shots.notna()].sort_values('shots')
        ax[0].errorbar(dd.shots, 100 * dd.f1, yerr=100 * dd.f1_sd, fmt=mk, color=col, capsize=3, label=cond)
    for cond, ls, col in [('exact simulator', '-', 'k'), ('classical matched', ':', 'gray')]:
        ax[0].axhline(100 * HWR[HWR.condition == cond].f1.iloc[0], ls=ls, color=col, label=cond)
    ax[0].set_xscale('log', base=2); ax[0].set_xlabel('shots per image'); ax[0].set_ylabel('macro-F1 (%)')
    ax[0].set_title(f"{n_hw} locked-test images on {info['backend']}"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].scatter(EVX.ravel(), EV_HW.ravel(), s=3, alpha=.3, color='#c0504d')
    ax[1].plot([-1, 1], [-1, 1], 'k--', lw=1); ax[1].plot([-1, 1], [-slope, slope], color='#c0504d', lw=1)
    ax[1].set_xlabel('exact <Y>'); ax[1].set_ylabel('measured <Y>'); ax[1].set_title(f'attenuation {slope:.3f}, r = {corr:.3f}')
    plt.tight_layout(); plt.savefig(f'{OUT}/hw_results.png', dpi=150); plt.show()

    hw = HWR[(HWR.condition == 'IBM hardware') & (HWR.shots == full_s)].iloc[0]
    ex = HWR[HWR.condition == 'exact simulator'].iloc[0]
    low = HWR[(HWR.condition == 'IBM hardware') & (HWR.shots == 16)]
    para = (f"We executed the trained hybrid VQC head on {info['backend']} for {n_hw} class-balanced locked-test images "
            f"({len(ug)} patients). The four 5-qubit sub-circuits were packed into one 20-qubit circuit; after transpilation it contained "
            f"{sum(info['two_qubit_gates'].values())} two-qubit gates ({info['logical_cnots']} logical CNOTs; two-qubit depth {info['depth_2q']}), "
            f"median two-qubit error {info['median_2q_error']}, median readout error {info['median_readout_error']}. "
            f"With {full_s} shots, hardware macro-F1 was {100*hw.f1:.1f}% (95% CI {100*hw.f1_lo:.1f}-{100*hw.f1_hi:.1f}) versus "
            f"{100*ex.f1:.1f}% for exact simulation (difference {SUMMARY['delta_f1_hw_minus_exact_pp']:+.1f} pp, 95% CI "
            f"{SUMMARY['delta_ci_pp'][0]:+.1f} to {SUMMARY['delta_ci_pp'][1]:+.1f}; patient-clustered McNemar p = {p_mc:.2g}); "
            f"{100*hw.agree_exact:.1f}% of hardware predictions matched the exact predictions. "
            + (f"With 16 shots, macro-F1 was {100*low.f1.iloc[0]:.1f} +- {100*low.f1_sd.iloc[0]:.1f}%. " if len(low) else '')
            + f"Measured expectation values were attenuated by a factor {slope:.2f} (r = {corr:.2f}), equivalent to uniform per-gate "
            f"depolarising noise p ~ {p_eff:.3f} in our simulator. No error mitigation was applied beyond dynamical decoupling.")
    print('\n' + para); open(f'{OUT}/hw_paper_paragraph.txt', 'w').write(para + '\n')
    if IBM_FAKE:
        print('\nFAKE BACKEND -- not hardware results.')

In [ ]:
# Cell 26: error mitigation -- unmitigated (Sampler, DD only) vs mitigated (Estimator: TREX + gate twirling + DD)
_mit = next((f for f in [f'{OUT}/hw_ev_mitigated.npy', f'{OUT}/hw_ev_mit_test.npy'] if os.path.exists(f)), None)
if _mit is None or not os.path.exists(f'{OUT}/hw_bits_main.npy'):
    print('needs the unmitigated main run (hw_bits_main.npy) and a mitigated run (hw_ev_mitigated.npy or hw_ev_mit_test.npy)')
else:
    _mode = 'mitigated' if _mit.endswith('hw_ev_mitigated.npy') else 'mit_test'
    job_m = json.load(open(f'{OUT}/ibm_job_{_mode}.json')) if os.path.exists(f'{OUT}/ibm_job_{_mode}.json') else {}
    st_m = job_m.get('settings', {})
    mit_label = 'mitigated (TREX + gate twirling + DD' + (' + ZNE' if st_m.get('zne') else '') + ')'
    EV_MIT = np.load(_mit); n_m = len(EV_MIT)
    HW_BACKEND = json.load(open(next(f for f in [f'{OUT}/hw_info_main.json', f'{OUT}/hw_info.json'] if os.path.exists(f))))['backend']
    EV_RAW = 1 - 2 * np.load(f'{OUT}/hw_bits_main.npy').mean(1)[:n_m]
    EVX, y_, g_ = EV_EXACT[:n_m], yS[:n_m], gS[:n_m]
    r_ = np.random.RandomState(99); ug, inv = np.unique(g_, return_inverse=True)
    mem = [np.where(inv == k)[0] for k in range(len(ug))]
    BOOTS = [np.concatenate([mem[k] for k in r_.randint(0, len(ug), len(ug))]) for _ in range(CFG['n_boot'])]
    # mitigated estimates can leave [-1, 1]; the head was trained on true expectation values, so clip before classifying
    CONDS = {'exact simulator': EVX, 'hardware, unmitigated (DD only)': EV_RAW, f'hardware, {mit_label}': np.clip(EV_MIT, -1, 1)}
    P = {k: head_from_ev(v) for k, v in CONDS.items()}
    pe = P['exact simulator'].argmax(1); rows = []
    for k, v in CONDS.items():
        pr = P[k].argmax(1); bf = [f1_score(y_[i], pr[i], average='macro') for i in BOOTS]
        raw = EV_MIT if k.startswith('hardware, mitigated') else v
        rows.append(dict(condition=k, f1=f1_score(y_, pr, average='macro'), f1_lo=np.percentile(bf, 2.5), f1_hi=np.percentile(bf, 97.5),
                         acc=accuracy_score(y_, pr), agree_exact=(pr == pe).mean(),
                         attenuation_slope=float((raw * EVX).sum() / (EVX ** 2).sum()),
                         pearson_r=float(np.corrcoef(raw.ravel(), EVX.ravel())[0, 1]), mae=float(np.abs(raw - EVX).mean())))
    MIT = pd.DataFrame(rows).set_index('condition'); MIT.to_csv(f'{OUT}/hw_mitigation_comparison.csv')
    print(f'{n_m} images, {len(ug)} patients'); print(MIT.round(4).to_string())

    def paired(a_name, b_name):
        a_, b_ = P[a_name].argmax(1), P[b_name].argmax(1)
        d = np.array([f1_score(y_[i], a_[i], average='macro') - f1_score(y_[i], b_[i], average='macro') for i in BOOTS])
        n01, n10 = int(((a_ == y_) & (b_ != y_)).sum()), int(((a_ != y_) & (b_ == y_)).sum())
        return dict(comparison=f'{a_name}  vs  {b_name}', delta_f1_pp=100 * (f1_score(y_, a_, average='macro') - f1_score(y_, b_, average='macro')),
                    ci_lo_pp=100 * np.percentile(d, 2.5), ci_hi_pp=100 * np.percentile(d, 97.5), first_only_correct=n01,
                    second_only_correct=n10, p_mcnemar=stats.binomtest(min(n01, n10), n01 + n10, .5).pvalue if n01 + n10 else 1.0,
                    p_mcnemar_clustered=clustered_mcnemar(a_ == y_, b_ == y_, g_), p_boot=boot_p(d))
    keys = list(CONDS)
    MT = pd.DataFrame([paired(keys[2], keys[0]), paired(keys[2], keys[1]), paired(keys[1], keys[0])])
    print(MT.round(4).to_string(index=False)); MT.to_csv(f'{OUT}/hw_mitigation_tests.csv', index=False)

    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for k, col in [(keys[1], '#9e9e9e'), (keys[2], '#2f6fb3')]:
        raw = EV_MIT if k == keys[2] else CONDS[k]
        ax[0].scatter(EVX.ravel(), raw.ravel(), s=3, alpha=.35, color=col, label=f"{k.split(', ')[1]}: slope {MIT.loc[k, 'attenuation_slope']:.2f}")
    ax[0].plot([-1, 1], [-1, 1], 'k--', lw=1); ax[0].set_xlabel('exact <Y>'); ax[0].set_ylabel('measured <Y>'); ax[0].legend(fontsize=8)
    ax[0].set_title('Expectation values: unmitigated vs mitigated')
    ax[1].errorbar(range(3), 100 * MIT.f1, yerr=[100 * (MIT.f1 - MIT.f1_lo), 100 * (MIT.f1_hi - MIT.f1)], fmt='o', capsize=4, color='k')
    ax[1].set_xticks(range(3)); ax[1].set_xticklabels(['exact', 'unmitigated', 'mitigated']); ax[1].set_ylabel('macro-F1 (%)')
    ax[1].set_title(f'Same {n_m} images on {HW_BACKEND}'); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{OUT}/hw_mitigation.png', dpi=150); plt.show()

    m, u, e = MIT.loc[keys[2]], MIT.loc[keys[1]], MIT.loc[keys[0]]
    para = (f"Error mitigation. Re-running the same {n_m} images on the same physical qubits with TREX readout mitigation, Pauli twirling of the "
            f"two-qubit gates and dynamical decoupling{' and zero-noise extrapolation' if st_m.get('zne') else ''} changed the attenuation of the "
            f"measured expectation values from {u.attenuation_slope:.2f} to {m.attenuation_slope:.2f} (Pearson r {u.pearson_r:.2f} -> {m.pearson_r:.2f}). "
            f"Macro-F1 was {100*m.f1:.1f}% (95% CI {100*m.f1_lo:.1f}-{100*m.f1_hi:.1f}) mitigated, {100*u.f1:.1f}% unmitigated and {100*e.f1:.1f}% exact "
            f"(mitigated - exact {MT.iloc[0].delta_f1_pp:+.1f} pp, 95% CI {MT.iloc[0].ci_lo_pp:+.1f} to {MT.iloc[0].ci_hi_pp:+.1f}; "
            f"mitigated - unmitigated {MT.iloc[1].delta_f1_pp:+.1f} pp, McNemar p = {MT.iloc[1].p_mcnemar:.2g}); "
            f"{100*m.agree_exact:.1f}% of mitigated predictions matched the exact ones.")
    print('\n' + para); open(f'{OUT}/hw_mitigation_paragraph.txt', 'w').write(para + '\n')
    if IBM_FAKE:
        print('\nFAKE BACKEND -- not hardware results.')

## External validation (no retraining)

The backbone and heads trained on Kermany are applied **unchanged** to independent OCT datasets from other centres and devices. Labels are mapped onto Kermany's four classes.

| Dataset | Device / centre | Classes used → Kermany class | How to get it |
|---|---|---|---|
| **OCTDL** (Kulyabin et al., *Sci. Data* 2024) | Optovue Avanti RTVue XR | NO → NORMAL, DME → DME, early and intermediate AMD (drusen) → DRUSEN; late AMD (geographic atrophy and neovascular AMD mixed) and ERM, RAO, RVO, VID excluded | Mendeley Data, DOI 10.17632/sncdhf53xc → *Download All* → put the zip in the external folder |
| **Duke / Srinivasan 2014** (*Biomed. Opt. Express*) | Spectralis, Duke | NORMAL → NORMAL, dry AMD → DRUSEN, DME → DME (45 volumes, 15 per class) | Downloaded automatically from duke.edu if reachable; otherwise download `2014_BOE_Srinivasan.zip` yourself and put it in the external folder |

The external folder is `MyDrive/oct_qml/external` on Colab; on Kaggle, add the zips as a dataset input instead.

**Scoring.**
- *4-way* (primary): the model chooses among all four classes, and a prediction of a class absent from the dataset counts as an error.
- *Restricted*: argmax over the classes present only.
- Macro-F1 is over the classes present, at image level, with patient-cluster bootstrap 95% CIs. A patient-level majority (mean probability per patient or volume) is also reported.
- Pre-declared comparison: VQC ensemble vs parameter-matched classical ensemble, 4-way macro-F1, patient-clustered McNemar test plus patient-cluster bootstrap CI, Holm-corrected across datasets.
- Kermany images are excluded by path and by file-name pattern, and an assertion checks that no external image is a Kermany file.

In [ ]:
# Cell 27: locate the external datasets (unzip / download if needed) and index them with patient IDs and mapped labels
import re, zipfile, urllib.request
EXT, EXT_INFO = {}, {}
if not RUN_EXTERNAL:
    print('RUN_EXTERNAL = False -> skipped')
else:
    EXT_DIR = os.environ.get('EXT_DIR') or os.path.join(os.path.dirname(OUT), 'external')
    EXT_WORK = '/content/ext' if os.path.isdir('/content') else os.path.join(os.path.dirname(OUT), 'ext_unzipped')
    os.makedirs(EXT_DIR, exist_ok=True); os.makedirs(EXT_WORK, exist_ok=True)
    IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

    # never pick up the Kermany training/test data itself (it also has a folder called DME)
    _KDIRS = {os.path.normpath(ROOT)}
    _parent = os.path.dirname(os.path.normpath(ROOT))
    if _parent not in ('/', '/kaggle/input', '/content', '/content/data') and os.path.basename(_parent) not in ('input', 'data'):
        _KDIRS.add(_parent)                                   # e.g. .../kermany2018 holding OCT2017/ and a duplicate copy
    KERMANY_FILE = re.compile(r'^(CNV|DME|DRUSEN|NORMAL)-\d+-\d+\.jpe?g$', re.I)

    def _in_kermany(d):
        d = os.path.normpath(d)
        return any(d == k or d.startswith(k + os.sep) for k in _KDIRS)

    def _walk_imgs(root):
        for d, _, fs in os.walk(root):
            if '__MACOSX' in d or _in_kermany(d):
                continue
            for f in fs:
                if f.lower().endswith(IMG_EXT) and not KERMANY_FILE.match(f):
                    yield os.path.join(d, f)

    def _have_duke(roots):
        return any(re.fullmatch(r'(AMD|DME|NORMAL)\d+', os.path.basename(d), re.I) for r in roots for d, _, _ in os.walk(r))

    roots = [EXT_DIR, EXT_WORK] + (['/kaggle/input'] if os.path.isdir('/kaggle/input') else [])
    if not _have_duke(roots):                              # Duke dataset: try the public URL once
        url = 'https://people.duke.edu/~sf59/Datasets/2014_BOE_Srinivasan.zip'
        dst = os.path.join(EXT_DIR, '2014_BOE_Srinivasan.zip')
        if not os.path.exists(dst):
            try:
                print('downloading the Duke/Srinivasan dataset ...'); urllib.request.urlretrieve(url, dst)
            except Exception as e:
                print(f'could not download {url} ({type(e).__name__}); download it manually into {EXT_DIR} to include it')
                if os.path.exists(dst):
                    os.remove(dst)
    for z in glob.glob(os.path.join(EXT_DIR, '*.zip')):
        tgt = os.path.join(EXT_WORK, os.path.splitext(os.path.basename(z))[0])
        if not os.path.exists(os.path.join(tgt, '.done')):
            print('unzipping', os.path.basename(z)); zipfile.ZipFile(z).extractall(tgt); open(os.path.join(tgt, '.done'), 'w').close()
            for inner in glob.glob(os.path.join(tgt, '**', '*.zip'), recursive=True):     # zips inside zips (Mendeley 'Download All')
                zipfile.ZipFile(inner).extractall(os.path.dirname(inner))

    # ---- OCTDL: folders AMD / DME / NO ..., files <label>_<patient>_<n>.jpg, metadata in OCTDL_labels.csv
    oct_imgs = [p for r in roots for p in _walk_imgs(r)
                if os.path.basename(os.path.dirname(p)).upper() in {'AMD', 'DME', 'NO', 'ERM', 'RAO', 'RVO', 'VID'}]
    if oct_imgs:
        csvs = [p for r in roots for p in glob.glob(os.path.join(r, '**', 'OCTDL_labels.csv'), recursive=True)]
        meta = pd.read_csv(csvs[0]) if csvs else None
        rows = []
        for p in sorted(set(oct_imgs)):
            folder = os.path.basename(os.path.dirname(p)).upper(); stem = os.path.splitext(os.path.basename(p))[0]
            parts = stem.split('_'); pid = parts[1] if len(parts) >= 3 else stem
            rows.append(dict(path=p, file=stem, disease=folder, patient=f'octdl_{pid}'))
        df = pd.DataFrame(rows)
        if meta is not None:
            meta = meta.copy(); meta['file'] = meta['file_name'].astype(str).map(lambda s: os.path.splitext(os.path.basename(s))[0])
            n0 = len(df)
            df = df.merge(meta[['file', 'subcategory', 'patient_id']], on='file', how='inner')   # only images listed in OCTDL_labels.csv
            print(f'OCTDL: {len(df)} of {n0} images found in OCTDL_labels.csv')
            df['patient'] = 'octdl_' + df.patient_id.astype(str)
        else:
            df['subcategory'] = np.nan
            print('OCTDL_labels.csv not found -> AMD cannot be split into drusen / MNV and is excluded')
        def _map(r):
            if r.disease == 'NO':  return 'NORMAL'
            if r.disease == 'DME': return 'DME'
            if r.disease == 'AMD':
                sub = str(r.subcategory).lower()
                # OCTDL grades AMD as early / intermediate / late. Early and intermediate AMD are defined by drusen
                # -> DRUSEN. Late AMD mixes geographic atrophy and neovascular AMD, so it cannot be mapped to CNV -> excluded.
                if sub in ('early', 'intermediate') or 'drusen' in sub: return 'DRUSEN'
                if any(k in sub for k in ('mnv', 'cnv', 'wet', 'neovasc')): return 'CNV'
            return None
        df['mapped'] = df.apply(_map, axis=1)
        print('OCTDL disease x subcategory -> mapped class (check this mapping):')
        print(df.groupby(['disease', df.subcategory.fillna('-'), df.mapped.fillna('excluded')]).size().to_string())
        df = df[df.mapped.notna()].copy()
        df['label'] = df.mapped.map({c: i for i, c in enumerate(CLASSES)})
        if len(df):
            EXT['OCTDL'] = df[['path', 'label', 'patient']].reset_index(drop=True)
            EXT_INFO['OCTDL'] = 'Optovue Avanti RTVue XR (OCTDL)'

    # ---- Duke / Srinivasan: volume folders AMD1..15, DME1..15, NORMAL1..15 (dry AMD -> DRUSEN); prefer 8-bit TIFFs
    rows = []
    for r in roots:
        for p in _walk_imgs(r):
            comp = next((c for c in p.split(os.sep) if re.fullmatch(r'(AMD|DME|NORMAL)\d+', c, re.I)), None)
            if comp:
                cls_ = re.match(r'[A-Za-z]+', comp).group(0).upper()
                rows.append(dict(path=p, vol=comp.upper(), label=CLASSES.index({'AMD': 'DRUSEN', 'DME': 'DME', 'NORMAL': 'NORMAL'}[cls_]),
                                 is8=('8bit' in p.lower())))
    if rows:
        df = pd.DataFrame(rows).drop_duplicates('path')
        has8 = df.groupby('vol').is8.transform('any')
        df = df[~has8 | df.is8]                                   # keep 8-bit TIFFs where both exist
        df['patient'] = 'duke_' + df.vol
        EXT['DUKE'] = df[['path', 'label', 'patient']].sort_values('path').reset_index(drop=True)
        EXT_INFO['DUKE'] = 'Spectralis, Duke (Srinivasan 2014)'

    # safety check: no external image may be a Kermany image
    _kerm = set(train_all.path.map(os.path.basename)) | set(official.path.map(os.path.basename))
    for k, v in EXT.items():
        _bad = v.path.map(os.path.basename).isin(_kerm).sum()
        assert _bad == 0, f'{k}: {_bad} images are Kermany files -- external set contaminated'
    if not EXT:
        print(f'No external dataset found. Put OCTDL / Duke zips in {EXT_DIR} (see the markdown above) and re-run.')
    for k, v in EXT.items():
        print(f"{k}: {len(v)} images, {v.patient.nunique()} patients/volumes |",
              dict(zip([CLASSES[i] for i in sorted(v.label.unique())], np.bincount(v.label, minlength=K)[sorted(v.label.unique())])))

In [ ]:
# Cell 28: features from the frozen backbone, head ensembles (5 folds x EXT_SEEDS), and evaluation
if not RUN_EXTERNAL or not EXT:
    print('nothing to evaluate')
else:
    CKPT = f"{OUT}/backbone_{CFG['backbone'].replace('.', '_')}.pt"
    assert os.path.exists(CKPT), f'backbone checkpoint not found ({CKPT}); the external features must come from the same backbone'
    bb = timm.create_model(CFG['backbone'], pretrained=False, num_classes=K, drop_rate=0.2).to(DEVICE).to(memory_format=torch.channels_last)
    bb.load_state_dict(torch.load(CKPT, map_location=DEVICE)); bb.eval()

    @torch.no_grad()
    def ext_features(df, bs=128):
        fs, ps = [], []
        for x, _ in make_loader(df, bs=bs):
            x = x.to(DEVICE).to(memory_format=torch.channels_last)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                f = bb.forward_head(bb.forward_features(x), pre_logits=True); lg = bb.get_classifier()(f)
            fs.append(f.float().cpu()); ps.append(torch.softmax(lg.float(), 1).cpu())
        return torch.cat(fs).numpy(), torch.cat(ps).numpy()

    EXT_FEAT = {}
    for name, df in EXT.items():
        cache = f'{OUT}/ext_features_{name}.npz'
        if os.path.exists(cache) and len(np.load(cache)['y']) == len(df) and (np.load(cache)['y'] == df.label.values).all():
            z = np.load(cache); X, PB = z['X'], z['bb_prob']
        else:
            t0 = time.time(); X, PB = ext_features(df); np.savez_compressed(cache, X=X, bb_prob=PB, y=df.label.values)
            print(f'{name}: features {X.shape} in {time.time() - t0:.0f}s')
        EXT_FEAT[name] = dict(X=X, bb_prob=PB, y=df.label.values, g=df.patient.values)
    EXT_FEAT['locked_test (Kermany)'] = dict(X=FEAT['locked_test']['X'], bb_prob=FEAT['locked_test']['bb_prob'],
                                             y=FEAT['locked_test']['y'], g=FEAT['locked_test']['g'])

    # ---- head ensembles: reuse the CV models when RUN_BENCHMARK ran them, else train them here (cached on disk)
    Xh, yh, gh = FEAT['head']['X'], FEAT['head']['y'], FEAT['head']['g']
    EXT_FOLDS = list(StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=GLOBAL_SEED).split(Xh, yh, gh))
    EXT_HEADS = ['bottleneck_only', 'classical_matched', 'hybrid_vqc']
    ens_file = f'{OUT}/ext_head_ensembles.pt'
    ENS = {}
    if 'MODELS' in globals() and all((h, 0, 0) in MODELS for h in EXT_HEADS):
        ENS = {k: v for k, v in MODELS.items() if k[0] in EXT_HEADS + ['logreg']}; ESC = SCALERS
        print('using the CV models from cell 15')
    else:
        saved = torch.load(ens_file, map_location=DEVICE, weights_only=False) if os.path.exists(ens_file) else None
        if saved is not None and saved.get('seeds') == list(EXT_SEEDS) and saved.get('n_folds') == CFG['n_folds']:
            ESC = {f: s for f, s in saved['scalers'].items()}
            for key, sd in saved['heads'].items():
                m = Head(key[0], FEAT_DIM, 0).to(DEVICE); m.load_state_dict(sd); m.eval(); ENS[key] = m
            for key, lr in saved['logreg'].items():
                ENS[key] = lr
            print('loaded cached head ensembles from', ens_file)
        else:
            ESC, store = {}, {'heads': {}, 'logreg': {}, 'scalers': {}, 'seeds': list(EXT_SEEDS), 'n_folds': CFG['n_folds']}
            for fold, (tri, vai) in enumerate(EXT_FOLDS):
                sc = StandardScaler().fit(Xh[tri]); ESC[fold] = sc; store['scalers'][fold] = sc
                Xtr, Xva = sc.transform(Xh[tri]), sc.transform(Xh[vai])
                lr, _ = fit_logreg(Xtr, yh[tri], Xva, yh[vai]); ENS[('logreg', fold, 0)] = lr; store['logreg'][('logreg', fold, 0)] = lr
                for s in EXT_SEEDS:
                    for h in EXT_HEADS:
                        m, _ = train_head(h, Xtr, yh[tri], Xva, yh[vai], seed=1000 * fold + s)
                        ENS[(h, fold, s)] = m; store['heads'][(h, fold, s)] = m.state_dict()
                print(f'fold {fold}: heads trained')
            torch.save(store, ens_file); print('saved', ens_file)

    def ens_probs(h, X):
        ps = []
        for (hh, fold, s), m in ENS.items():
            if hh == h:
                Xs = ESC[fold].transform(X)
                ps.append(m.predict_proba(Xs) if h == 'logreg' else predict_head(m, Xs))
        return np.mean(ps, 0)

    MODELS_EXT = ['backbone_e2e', 'logreg'] + EXT_HEADS
    NAME = {'backbone_e2e': 'ResNet end-to-end', 'logreg': 'Logistic regression', 'bottleneck_only': 'Bottleneck only',
            'classical_matched': 'Classical, 120 params', 'hybrid_vqc': 'VQC, 120 params'}
    def macro_f1_present(y, pred, present):
        return f1_score(y, pred, labels=present, average='macro', zero_division=0)

    ROWS, TESTS, PROBS_EXT = [], [], {}
    for dname, d in EXT_FEAT.items():
        y, g = d['y'], d['g']; present = sorted(np.unique(y)); mask = np.zeros(K, bool); mask[present] = True
        ug, inv = np.unique(g, return_inverse=True); mem = [np.where(inv == k)[0] for k in range(len(ug))]
        rb = np.random.RandomState(321)
        boots = [np.concatenate([mem[k] for k in rb.randint(0, len(ug), len(ug))]) for _ in range(CFG['n_boot'])]
        P = {h: (d['bb_prob'] if h == 'backbone_e2e' else ens_probs(h, d['X'])) for h in MODELS_EXT}; PROBS_EXT[dname] = P
        for h, p in P.items():
            pred4 = p.argmax(1); predR = np.where(mask, p, -1).argmax(1)
            bf = [macro_f1_present(y[i], pred4[i], present) for i in boots]
            # patient / volume level: mean probability per patient, then argmax (4-way)
            pp = np.stack([p[inv == k].mean(0) for k in range(len(ug))]); yp = np.array([np.bincount(y[inv == k]).argmax() for k in range(len(ug))])
            ROWS.append(dict(dataset=dname, model=NAME[h], images=len(y), patients=len(ug),
                             f1_4way=macro_f1_present(y, pred4, present), f1_lo=np.percentile(bf, 2.5), f1_hi=np.percentile(bf, 97.5),
                             acc_4way=accuracy_score(y, pred4), f1_restricted=macro_f1_present(y, predR, present),
                             patient_acc=accuracy_score(yp, pp.argmax(1)), patient_f1=macro_f1_present(yp, pp.argmax(1), present)))
        a_, b_ = P['hybrid_vqc'].argmax(1), P['classical_matched'].argmax(1)
        dlt = np.array([macro_f1_present(y[i], a_[i], present) - macro_f1_present(y[i], b_[i], present) for i in boots])
        n01, n10 = int(((a_ == y) & (b_ != y)).sum()), int(((a_ != y) & (b_ == y)).sum())
        TESTS.append(dict(dataset=dname, delta_f1_vqc_minus_classical_pp=100 * (macro_f1_present(y, a_, present) - macro_f1_present(y, b_, present)),
                          ci_lo_pp=100 * np.percentile(dlt, 2.5), ci_hi_pp=100 * np.percentile(dlt, 97.5),
                          vqc_only_correct=n01, classical_only_correct=n10,
                          p_boot=boot_p(dlt),
                          p_mcnemar=clustered_mcnemar(a_ == y, b_ == y, g)))   # clustered by patient / volume
    EXTR = pd.DataFrame(ROWS); EXTT = pd.DataFrame(TESTS)
    ext_only = ~EXTT.dataset.str.startswith('locked_test')
    def _holm(p):
        p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
        for r_, i in enumerate(o):
            run = max(run, (len(p) - r_) * p[i]); adj[i] = min(1.0, run)
        return adj
    EXTT['p_holm'] = np.nan
    EXTT.loc[ext_only, 'p_holm'] = _holm(EXTT.loc[ext_only, 'p_mcnemar'].values)
    EXTR.to_csv(f'{OUT}/external_validation.csv', index=False); EXTT.to_csv(f'{OUT}/external_validation_tests.csv', index=False)
    show = EXTR.copy()
    for c in ['f1_4way', 'f1_lo', 'f1_hi', 'acc_4way', 'f1_restricted', 'patient_acc', 'patient_f1']:
        show[c] = (100 * show[c]).round(2)
    print(show.to_string(index=False)); print(); print(EXTT.round(4).to_string(index=False))

    fig, axs = plt.subplots(1, len(EXT_FEAT), figsize=(5.2 * len(EXT_FEAT), 4.4), squeeze=False)
    for ax, (dname, P) in zip(axs[0], PROBS_EXT.items()):
        y = EXT_FEAT[dname]['y']; present = sorted(np.unique(y))
        cm = confusion_matrix(y, P['hybrid_vqc'].argmax(1), labels=range(K))[present]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=[CLASSES[i] for i in present], ax=ax, cbar=False)
        ax.set_title(f'{dname}: VQC ensemble'); ax.set_xlabel('predicted'); ax.set_ylabel('true')
    plt.tight_layout(); plt.savefig(f'{OUT}/external_confusion.png', dpi=150); plt.show()

    lines = []
    for dname in [k for k in EXT_FEAT if not k.startswith('locked_test')]:
        r = EXTR[(EXTR.dataset == dname)].set_index('model'); t = EXTT[EXTT.dataset == dname].iloc[0]
        lt = EXTR[EXTR.dataset.str.startswith('locked_test')].set_index('model')
        lines.append(f"{dname} ({EXT_INFO[dname]}; {int(r.images.iloc[0])} images, {int(r.patients.iloc[0])} patients/volumes): "
                     f"VQC macro-F1 {100*r.loc['VQC, 120 params','f1_4way']:.1f}% vs classical {100*r.loc['Classical, 120 params','f1_4way']:.1f}% "
                     f"(diff {t.delta_f1_vqc_minus_classical_pp:+.1f} pp, 95% CI {t.ci_lo_pp:+.1f} to {t.ci_hi_pp:+.1f}, Holm p (clustered McNemar) = {t.p_holm:.2g}, bootstrap p = {t.p_boot:.2g}); "
                     f"drop from the Kermany locked test: VQC {100*(lt.loc['VQC, 120 params','f1_4way']-r.loc['VQC, 120 params','f1_4way']):.1f} pp, "
                     f"classical {100*(lt.loc['Classical, 120 params','f1_4way']-r.loc['Classical, 120 params','f1_4way']):.1f} pp.")
    para = 'External validation without retraining. ' + ' '.join(lines)
    print('\n' + para); open(f'{OUT}/external_paragraph.txt', 'w').write(para + '\n')

In [ ]:
# Cell 29: EXPLORATORY (not pre-declared) -- robustness of the external comparisons to training randomness, per-class F1, prediction mix
if not RUN_EXTERNAL or not EXT:
    print('nothing to evaluate')
else:
    from scipy.stats import wilcoxon
    src_models = MODELS if 'MODELS' in globals() else ENS          # CV models of Cell 15, else the cached ensembles of Cell 28
    src_scal = SCALERS if 'MODELS' in globals() else ESC

    def one_model_probs(key, X):
        Xs = src_scal[key[1]].transform(X)
        return src_models[key].predict_proba(Xs) if key[0] == 'logreg' else predict_head(src_models[key], Xs)

    def f1p(y, pred, present):
        return f1_score(y, pred, labels=present, average='macro', zero_division=0)

    ROB, PERCLASS, PREDDIST = [], [], []
    rpairs = [('hybrid_vqc', 'classical_matched'), ('hybrid_vqc', 'bottleneck_only')]
    rpairs += [p for p in [(f'reupload_L{L}', f'classical_deep_{2 * L // 3}') for L in CFG['reupload_layers']]
               if all((h, 0, CFG['head_seeds'][0]) in src_models for h in p)]
    rheads = sorted({h for p in rpairs for h in p})
    keys = sorted({(f, s) for (h, f, s) in src_models if h == 'hybrid_vqc'})
    for dname, d in EXT_FEAT.items():
        y, X = d['y'], d['X']; present = sorted(np.unique(y))
        probs = {h: {k: one_model_probs((h,) + k, X) for k in keys} for h in rheads}
        # (1) every single model, paired by fold and seed
        for qa, cb in rpairs:
            fq = np.array([f1p(y, probs[qa][k].argmax(1), present) for k in keys])
            fc = np.array([f1p(y, probs[cb][k].argmax(1), present) for k in keys])
            dif = fq - fc
            ROB.append(dict(dataset=dname, pair=f'{qa} - {cb}', models=len(keys),
                            mean_single_f1_q=100 * fq.mean(), mean_single_f1_c=100 * fc.mean(),
                            mean_diff_pp=100 * dif.mean(), sd_diff_pp=100 * dif.std(ddof=1),
                            quantum_better=f'{(dif > 0).sum()}/{len(dif)}',
                            p_sign=stats.binomtest(int((dif > 0).sum()), int((dif != 0).sum()), .5).pvalue if (dif != 0).any() else 1.0,
                            p_wilcoxon=wilcoxon(dif).pvalue if (dif != 0).any() else 1.0))
        # (2) per-class F1 and confidence of the ensembles, (3) share of images predicted as each class
        for h in rheads:
            pr = np.mean(list(probs[h].values()), 0); pred = pr.argmax(1)
            f1c = f1_score(y, pred, labels=present, average=None, zero_division=0)
            PERCLASS.append(dict(dataset=dname, head=h, **{CLASSES[c]: round(100 * v, 2) for c, v in zip(present, f1c)},
                                 macro=round(100 * f1c.mean(), 2), mean_max_prob=round(pr.max(1).mean(), 3)))
            PREDDIST.append(dict(dataset=dname, head=h, **{f'pred_{c}': round(100 * (pred == i).mean(), 1) for i, c in enumerate(CLASSES)}))
        PREDDIST.append(dict(dataset=dname, head='TRUE', **{f'pred_{c}': round(100 * (y == i).mean(), 1) for i, c in enumerate(CLASSES)}))

    ROB, PERCLASS, PREDDIST = pd.DataFrame(ROB), pd.DataFrame(PERCLASS), pd.DataFrame(PREDDIST)
    ROB.to_csv(f'{OUT}/external_robustness_single_models.csv', index=False)
    PERCLASS.to_csv(f'{OUT}/external_per_class_f1.csv', index=False); PREDDIST.to_csv(f'{OUT}/external_prediction_distribution.csv', index=False)
    print('(1) single models, paired by fold x seed (macro-F1 %, classes present):'); print(ROB.round(4).to_string(index=False))
    print('\n(2) per-class F1 of the ensembles (%), and mean confidence:'); print(PERCLASS.to_string(index=False))
    print('\n(3) share of images predicted as each class (%), TRUE = actual class mix:'); print(PREDDIST.to_string(index=False))

In [ ]:
# Cell 30: record the software environment of this run (for the supplement / reviewers)
import platform, subprocess
env = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
with open(f'{OUT}/environment_02.txt', 'w') as f:
    f.write(f'python {platform.python_version()} | torch {torch.__version__} | cuda {torch.version.cuda} | device {gpu}\n\n' + env)
print('environment written to', f'{OUT}/environment_02.txt')